In [ ]:
%%time
import os
import socket
from pathlib import Path
import pandas as pd
import numpy as np

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

try:
    from server_timezones import utc_offset_hours as _server_utc_offset_hours
except Exception:
    # Fallback keeps the notebook runnable if this file is copied away from the repo.
    _FALLBACK_UTC_OFFSET_HOURS = {
        "ID1": 7,
        "EUW1": 1,
        "NA1": -5,
        "EUN1": 2,
        "BR1": -3,
        "TR1": 3,
        "JP1": 9,
        "LA1": -6,
        "LA2": -3,
        "OC1": 10,
        "PBE1": -8,
    }

    def _server_utc_offset_hours(server):
        return _FALLBACK_UTC_OFFSET_HOURS[server]


def platform_utc_offset_hours(server):
    """Return the fixed UTC offset used for local-hour plots."""
    try:
        return int(_server_utc_offset_hours(server))
    except KeyError:
        print(f"No UTC offset configured for {server}; using UTC+0.")
        return 0


# Detect environment and use a valid parquet path without hard dependency on Colab
hostname = socket.gethostname().lower()
colab_db_path = "/content/drive/Shareddrives/MSc_2026_Riot/db/riotData.parquet"
local_db_path = "/raid/data/riot/riotData.parquet"

if Path(local_db_path).exists():
    db_path = local_db_path
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        db_path = colab_db_path
    except ImportError:
        # Fallback for teaching/demo runs where data may be in a custom location
        db_path = os.environ.get("RIOT_DB_PATH", local_db_path)
        if not Path(db_path).exists():
            raise FileNotFoundError(
                "Could not locate riotData parquet. Set RIOT_DB_PATH or place the file at "
                f"{local_db_path}"
            )

platform = "EUW1"  # <- change as needed
target_col = "TIMEPLAYED"
TOP_N_PLAYERS = 1000  # single cohort size used for player-level analyses

min_period_h = 6.0
max_period_h = 48.0
n_freq = 500

agg_col_map = {
    "NEUTRALCREEP": "neutralcreep_mean",
    "ENEMYCREEP": "enemycreep_mean",
    "GOLD": "gold_mean",
    "DAMDEALT": "damdealt_mean",
    "TIMEDEAD": "timedead_mean",
    "TIMEPLAYED": "timeplayed_mean",
    "KILLS": "kills_mean",
    "DEATHS": "deaths_mean",
    "ASSISTS": "assists_mean",
}

# Visual style used across all plots for a cohesive story.
COLORS = {
    "ink": "#264653",
    "primary": "#1d3557",
    "secondary": "#2a9d8f",
    "accent": "#e76f51",
    "highlight": "#f4a261",
    "muted": "#8d99ae",
    "panel": "#f8f5ef",
}

plt.rcParams.update({
    "figure.facecolor": "#fcfbf8",
    "axes.facecolor": COLORS["panel"],
    "axes.edgecolor": "#d9d1c8",
    "axes.titleweight": "bold",
    "axes.labelcolor": COLORS["ink"],
    "xtick.color": COLORS["ink"],
    "ytick.color": COLORS["ink"],
    "grid.color": "#cfc7bc",
    "grid.linestyle": "--",
    "grid.alpha": 0.28,
    "savefig.facecolor": "#fcfbf8",
})


def style_axes(ax, grid_axis="both"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#d9d1c8")
    ax.spines["bottom"].set_color("#d9d1c8")
    ax.grid(True, axis=grid_axis)
    return ax


def styled_hist(ax, data, bins=50, variant="secondary", **kwargs):
    """Draw a histogram using the shared notebook plot style."""
    hist_kwargs = {
        "color": COLORS.get(variant, COLORS["secondary"]),
        "alpha": 0.85,
        "edgecolor": "white",
    }
    hist_kwargs.update(kwargs)
    ax.hist(data, bins=bins, **hist_kwargs)
    style_axes(ax, grid_axis="y")
    return ax


In [ ]:
%%time
# DuckDB connection helper (replaces sqlite3 for parquet support)
import os
import duckdb
import pandas as pd
from pathlib import Path

# Colab persists the aggregate cache on shared Drive so later sessions can reuse it.
# RIOT_DUCKDB_PATH overrides the default, for example to use a personal Drive cache.
if Path('/content').exists():
    _default_duckdb_file = '/content/drive/Shareddrives/MSc_2026_Riot/db/riot_local.duckdb'
else:
    _default_duckdb_file = './riot_local.duckdb'
_duckdb_file = os.environ.get('RIOT_DUCKDB_PATH', _default_duckdb_file)

# Reuse a valid persistent cache read-only. Only the first build needs a writer.
_required_hourly_columns = {"platformid", "hour_idx", "n"} | {col.lower() for col in agg_col_map.values()}
_duck_conn = None
_duckdb_cache_read_only = False
if Path(_duckdb_file).exists():
    try:
        _candidate = duckdb.connect(_duckdb_file, read_only=True)
        _candidate.execute("SELECT 1 FROM riotData LIMIT 1").fetchone()
        _candidate.execute("SELECT 1 FROM hourly_agg LIMIT 1").fetchone()
        _existing_columns = {row[1].lower() for row in _candidate.execute("PRAGMA table_info('hourly_agg')").fetchall()}
        if _required_hourly_columns.issubset(_existing_columns):
            _duck_conn = _candidate
            _duckdb_cache_read_only = True
        else:
            _candidate.close()
    except Exception:
        try:
            _candidate.close()
        except Exception:
            pass

if _duck_conn is None:
    Path(_duckdb_file).parent.mkdir(parents=True, exist_ok=True)
    _duck_conn = duckdb.connect(_duckdb_file)

print(f"DuckDB file: {_duckdb_file}")
print(f"Reusing valid cache read-only: {_duckdb_cache_read_only}")

# Let DuckDB use multiple CPU threads for parquet scans/queries
duckdb_threads = max(1, (os.cpu_count() or 1) - 1)
_duck_conn.execute(f"PRAGMA threads={duckdb_threads}")
print(f"DuckDB threads set to: {duckdb_threads}")

# A new cache needs a view that reads directly from the parquet file.
if not _duckdb_cache_read_only:
    try:
        db_path_sql = str(db_path).replace("'", "''")
        _duck_conn.execute(f"CREATE OR REPLACE VIEW riotData AS SELECT * FROM read_parquet('{db_path_sql}')")
    except Exception as e:
        print(f"Could not create view on {db_path}: {e}")


class DBWrapper:
    def __init__(self, dconn):
        self.dconn = dconn

    def execute(self, sql, params=None):
        # Ignore sqlite-specific pragmas and index creation statements
        upper_sql = sql.upper()
        if "CREATE INDEX" in upper_sql or "PRAGMA TEMP_STORE" in upper_sql or "PRAGMA CACHE_SIZE" in upper_sql or "PRAGMA MMAP_SIZE" in upper_sql:
            return self

        if params:
            return self.dconn.execute(sql, params)
        return self.dconn.execute(sql)

    def commit(self):
        pass

    def close(self):
        pass

    def fetchall(self):
        return self.dconn.fetchall()


def connect_db(path: str):
    return DBWrapper(_duck_conn)


# Monkey-patch pandas to use DuckDB natively for our wrapper
_orig_read_sql_query = pd.read_sql_query


def duckdb_read_sql_query(sql, con, params=None, **kwargs):
    if isinstance(con, DBWrapper):
        if params:
            return con.dconn.execute(sql, params).df()
        return con.dconn.execute(sql).df()
    return _orig_read_sql_query(sql, con, params=params, **kwargs)


pd.read_sql_query = duckdb_read_sql_query


def duckdb_relation_exists(relation_name):
    """Return True if a table or view exists in the active DuckDB file."""
    existing = {row[0] for row in _duck_conn.execute("SHOW TABLES").fetchall()}
    return relation_name in existing


def duckdb_relation_columns(relation_name):
    """Return lower-case column names for a DuckDB table or view."""
    rows = _duck_conn.execute(f"PRAGMA table_info('{relation_name}')").fetchall()
    return {row[1].lower() for row in rows}


def build_hourly_agg_table():
    """Materialize hourly platform aggregates used by the later analysis cells."""
    _duck_conn.execute("DROP TABLE IF EXISTS hourly_agg")
    _duck_conn.execute(
        """
        CREATE TABLE hourly_agg AS
        SELECT
          PLATFORMID AS platformid,
          CAST(FLOOR(CAST(TIMESTAMP AS DOUBLE) / 3600000.0) AS BIGINT) AS hour_idx,
          AVG(CAST(NULLIF(NEUTRALCREEP, '') AS DOUBLE)) AS neutralcreep_mean,
          AVG(CAST(NULLIF(ENEMYCREEP, '') AS DOUBLE)) AS enemycreep_mean,
          AVG(CAST(NULLIF(GOLD, '') AS DOUBLE)) AS gold_mean,
          AVG(CAST(NULLIF(DAMDEALT, '') AS DOUBLE)) AS damdealt_mean,
          AVG(CAST(NULLIF(TIMEDEAD, '') AS DOUBLE)) AS timedead_mean,
          AVG(CAST(NULLIF(TIMEPLAYED, '') AS DOUBLE)) AS timeplayed_mean,
          AVG(CAST(NULLIF(KILLS, '') AS DOUBLE)) AS kills_mean,
          AVG(CAST(NULLIF(DEATHS, '') AS DOUBLE)) AS deaths_mean,
          AVG(CAST(NULLIF(ASSISTS, '') AS DOUBLE)) AS assists_mean,
          COUNT(*) AS n
        FROM riotData
        WHERE PLATFORMID IS NOT NULL
          AND TIMESTAMP IS NOT NULL
        GROUP BY PLATFORMID, hour_idx;
        """
    )
    _duck_conn.execute("CREATE INDEX IF NOT EXISTS idx_hourly_agg_platform_hour ON hourly_agg(platformid, hour_idx)")


def ensure_hourly_agg_table(rebuild=False):
    """Create hourly_agg when missing, or rebuild it when requested."""
    required_columns = {"platformid", "hour_idx", "n"} | {col.lower() for col in agg_col_map.values()}
    exists = duckdb_relation_exists("hourly_agg")

    if exists and not rebuild:
        existing_columns = duckdb_relation_columns("hourly_agg")
        if required_columns.issubset(existing_columns):
            print("hourly_agg table found; using existing aggregate table.")
            return False
        print("hourly_agg exists but is missing expected columns; rebuilding it.")

    if rebuild and exists:
        print("Rebuilding hourly_agg table from riotData. This may take a few minutes.")
    elif not exists:
        print("Building hourly_agg table from riotData. This may take a few minutes.")

    build_hourly_agg_table()
    print("hourly_agg rebuilt and indexed.")
    return True


In [ ]:
%%time
# Platform overview & quick hourly sample
conn = connect_db(db_path)

# Ensure the materialized hourly table exists before downstream cells use it.
ensure_hourly_agg_table(rebuild=False)

# Show row counts per platform (raw rows in riotData)
overview = pd.read_sql_query(
    """
    SELECT PLATFORMID, COUNT(*) AS n_rows
    FROM riotData
    WHERE PLATFORMID IS NOT NULL
    GROUP BY PLATFORMID
    ORDER BY n_rows DESC;
    """,
    conn,
)
print("Rows per PLATFORMID in riotData (raw rows):")
print(overview)

# Show hourly_agg coverage per platform.
hourly_overview = pd.read_sql_query(
    """
    SELECT
      platformid AS PLATFORMID,
      COUNT(*) AS n_hour_bins,
      SUM(n) AS n_rows
    FROM hourly_agg
    WHERE platformid IS NOT NULL
    GROUP BY platformid
    ORDER BY n_rows DESC;
    """,
    conn,
)
print("\nRows per PLATFORMID in hourly_agg (SUM(n) ~ underlying rows), plus number of hourly bins:")
print(hourly_overview)

print("\nDuckDB reads riotData as a parquet-backed view; no raw-table index is created.")

# Quick sample of hourly aggregated data (for selected platform only)
agg_col = agg_col_map[target_col]
hourly_query = f"""
SELECT
  hour_idx,
  {agg_col} AS target_mean,
  n
FROM hourly_agg
WHERE platformid = ?
ORDER BY hour_idx;
"""

hourly = pd.read_sql_query(hourly_query, conn, params=[platform])
conn.close()

print(f"\nUsing database: {db_path}")
print(f"Hourly bins for {platform}: {len(hourly)}")


## Why normalization matters
Before converting performance metrics into per-minute or per-second rates, check whether game duration itself changes with time of day.
If matches are longer at some hours than others, raw totals such as GOLD per game can look rhythmic even when player efficiency is unchanged.

In [ ]:
%%time
# Raw game duration by local start hour (before any normalization step)
offset_hours = platform_utc_offset_hours(platform)
print(f"Inspecting raw game duration by local hour using UTC{offset_hours:+d} for {platform}")

conn = connect_db(db_path)
duration_by_hour = pd.read_sql_query(
    """
    WITH raw_games AS (
        SELECT
            CAST(TIMESTAMP AS DOUBLE) / 3600000.0 AS start_hour_utc,
            CAST(NULLIF(TIMEPLAYED, '') AS DOUBLE) AS timeplayed_seconds
        FROM riotData
        WHERE PLATFORMID = ?
          AND TIMESTAMP IS NOT NULL
          AND TIMEPLAYED IS NOT NULL
          AND NULLIF(TIMEPLAYED, '') IS NOT NULL
    ),
    by_hour AS (
        SELECT
            CAST(FLOOR((start_hour_utc + ?) % 24.0) AS INTEGER) AS local_hour,
            timeplayed_seconds / 60.0 AS duration_minutes
        FROM raw_games
        WHERE timeplayed_seconds > 0
    )
    SELECT
        local_hour,
        COUNT(*) AS n_games,
        AVG(duration_minutes) AS mean_duration_minutes,
        quantile_cont(duration_minutes, 0.25) AS q25_duration_minutes,
        quantile_cont(duration_minutes, 0.5) AS median_duration_minutes,
        quantile_cont(duration_minutes, 0.75) AS q75_duration_minutes
    FROM by_hour
    GROUP BY local_hour
    ORDER BY local_hour;
    """,
    conn,
    params=[platform, offset_hours],
)
conn.close()

if duration_by_hour.empty:
    raise RuntimeError(f"No raw game durations found for PLATFORMID={platform}")

all_hours = pd.DataFrame({"local_hour": np.arange(24)})
duration_by_hour = all_hours.merge(duration_by_hour, on="local_hour", how="left")

overall_median_duration = duration_by_hour["median_duration_minutes"].median()
print(f"Median game duration across local hours: {overall_median_duration:.2f} minutes")

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 5.2),
    gridspec_kw={"width_ratios": [1.35, 1.0]},
)

ax = axes[0]
ax.fill_between(
    duration_by_hour["local_hour"],
    duration_by_hour["q25_duration_minutes"],
    duration_by_hour["q75_duration_minutes"],
    color=COLORS["secondary"],
    alpha=0.24,
    label="IQR",
)
ax.plot(
    duration_by_hour["local_hour"],
    duration_by_hour["median_duration_minutes"],
    color=COLORS["primary"],
    linewidth=2.4,
    marker="o",
    markersize=5,
    label="Median duration",
)
ax.plot(
    duration_by_hour["local_hour"],
    duration_by_hour["mean_duration_minutes"],
    color=COLORS["accent"],
    linewidth=1.8,
    linestyle="--",
    label="Mean duration",
)
ax.axhline(overall_median_duration, color=COLORS["muted"], linestyle=":", linewidth=1.4, label="Overall median")
ax.set_title(f"Raw Game Duration by Local Start Hour ({platform})")
ax.set_xlabel("Local start hour")
ax.set_ylabel("Game duration (minutes)")
ax.set_xticks(np.arange(0, 24, 2))
style_axes(ax, grid_axis="y")
ax.legend(frameon=False, loc="upper right")

ax = axes[1]
ax.bar(
    duration_by_hour["local_hour"],
    duration_by_hour["n_games"].fillna(0),
    color=COLORS["highlight"],
    alpha=0.88,
    edgecolor="white",
    linewidth=0.8,
    label="Games observed",
)
ax.set_title(f"Raw Game Counts by Local Start Hour ({platform})")
ax.set_xlabel("Local start hour")
ax.set_ylabel("Number of games")
ax.set_xticks(np.arange(0, 24, 2))
style_axes(ax, grid_axis="y")

fig.suptitle(
    f"Why raw totals can be misleading: PLATFORMID={platform}",
    fontsize=13.5,
    fontweight="bold",
    color=COLORS["ink"],
)
plt.tight_layout()
plt.show()

# Hourly aggregate table

The setup cells create `hourly_agg` if it is missing. Use the next cell only when you want to force a rebuild after changing the aggregate definition.


In [ ]:
%%time
# Hourly aggregate maintenance cell.
# Leave this False for normal top-to-bottom runs. Set True only to refresh the table.
REBUILD_HOURLY_AGG = False
ensure_hourly_agg_table(rebuild=REBUILD_HOURLY_AGG)


# Inspection

There are lots of code blocks like this that basically just examine existing data and tell you what's there.

In [ ]:
%%time
# Inspect available fields in riotData

conn = connect_db(db_path)
cols = conn.execute("PRAGMA table_info(riotData);").fetchall()
conn.close()

columns = [c[1] for c in cols]
print(f"riotData columns ({len(columns)}):")
print(columns)

# Summary

Similarly, here we look at summary stats - make sure everything is looking sane. We specifically check for NaNs (not a number) - these will impact later analysis so we need to remove them

In [ ]:
%%time
import pandas as pd
import numpy as np

print("=== Data Diagnostics & Summary Stats ===\n")
print("This cell summarizes DataFrames that already exist in the current kernel.")
print("Run it again later if you want diagnostics after PCA or cohort extraction.\n")

diagnostic_tables = {
    "hourly": globals().get("hourly"),
    "hourly_gold": globals().get("hourly_gold"),
    "top_players_data": globals().get("top_players_data"),
}

for table_name, df in diagnostic_tables.items():
    if not isinstance(df, pd.DataFrame):
        print(f"--- {table_name} ---")
        print("Not available yet.\n")
        continue

    print(f"--- {table_name} ---")
    print(f"Total Rows: {len(df)}")
    print("\nNaN Count per Column:")
    print(df.isna().sum())

    numeric_summary = df.describe(include=[np.number]).T
    if numeric_summary.empty:
        print("\nNo numeric columns to summarize.\n")
    else:
        print("\nSummary Statistics (Numeric):")
        display(numeric_summary)
        print()


# Data filtering

Cleaning, outlier rejection - for example, get rid of things beyond some limit like 2600 hours - 100 days or so

In [ ]:
MAX_HOUR_LIMIT=10000


In [ ]:
%%time
# Data cleaning: filter by continuous time window and outliers
if hourly.empty:
    raise RuntimeError(f"No rows for PLATFORMID={platform}")

# Filter to only include rows with data spread over ~MAX_HOUR_LIMIT hours
min_hour = hourly['hour_idx'].min()
max_hour = hourly['hour_idx'].max()
print(f"Before filtering: {len(hourly)} rows")
print(f"Hour_idx range: {min_hour} to {max_hour} (duration: {max_hour - min_hour} hours)")

duration_hours = max_hour - min_hour
if duration_hours > MAX_HOUR_LIMIT:
    cutoff_idx = min_hour + MAX_HOUR_LIMIT
    hourly = hourly[hourly['hour_idx'] <= cutoff_idx]
    print(f"After filtering to first {MAX_HOUR_LIMIT} hours: {len(hourly)} rows")

# Calculate IQR to identify outliers in hour_idx
Q1 = hourly['hour_idx'].quantile(0.25)
Q3 = hourly['hour_idx'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds and filter
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 3 * IQR
hourly_filtered = hourly[(hourly['hour_idx'] >= lower_bound) & (hourly['hour_idx'] <= upper_bound)]

print(f"After IQR filtering: {len(hourly_filtered)} rows (removed {len(hourly) - len(hourly_filtered)} outliers)")

hourly = hourly_filtered.copy()

# Periodogram: single metric (target_col) over time
t_hours = hourly["hour_idx"].to_numpy(dtype=float, copy=True)
t_hours -= t_hours.min()

y = hourly["target_mean"].to_numpy(dtype=float, copy=True)

# Drop NaNs before proceeding
valid_mask = ~np.isnan(y)
t_hours = t_hours[valid_mask]
y = y[valid_mask]

# Demean the valid data
y -= np.mean(y)

min_freq = 1.0 / max_period_h
max_freq = 1.0 / min_period_h
frequency = np.linspace(min_freq, max_freq, n_freq)

ls = LombScargle(t_hours, y)
power = ls.power(frequency)
period = 1.0 / frequency

best_idx = int(np.argmax(power))
best_period = float(period[best_idx])
power_24 = float(power[int(np.argmin(np.abs(period - 24.0)))])

print(f"Best period: {best_period:.3f} h")
print(f"Power near 24 h: {power_24:.5f}")

# Combined figure: time series + periodogram
fig, axes = plt.subplots(1, 2, figsize=(15, 5.4), gridspec_kw={"width_ratios": [1.15, 1.0]})

# Left panel: demeaned time series
ax = axes[0]
ax.plot(t_hours, y, linewidth=1.1, color=COLORS["primary"], alpha=0.9)
ax.fill_between(t_hours, y, 0, color=COLORS["primary"], alpha=0.12)
ax.set_title(f"{target_col} Over Time (Demeaned), PLATFORMID={platform}")
ax.set_xlabel("Hours Since Start")
ax.set_ylabel(f"Mean {target_col} (demeaned)")
style_axes(ax)

# Right panel: periodogram
ax = axes[1]
ax.plot(period, power, color=COLORS["secondary"], linewidth=2.0)
ax.fill_between(period, power, 0, color=COLORS["secondary"], alpha=0.2)
ax.axvline(24.0, linestyle="--", linewidth=1.4, color=COLORS["accent"], alpha=0.95, label="24 h")
if max_period_h >= 168:
    ax.axvline(168.0, linestyle="--", linewidth=1.2, color=COLORS["highlight"], alpha=0.75, label="168 h")
ax.axvline(best_period, linestyle=":", linewidth=1.4, color=COLORS["ink"], alpha=0.9, label=f"Peak {best_period:.1f} h")
ax.set_xlim(0, max_period_h)
ax.set_title(f"Lomb-Scargle Periodogram, PLATFORMID={platform}")
ax.set_xlabel("Period (hours)")
ax.set_ylabel("Power")
style_axes(ax)
ax.legend(loc="upper right", frameon=False)

fig.suptitle(f"Rhythm Overview: PLATFORMID={platform}", fontsize=14, fontweight="bold", color=COLORS["ink"])
plt.tight_layout()
plt.show()

# OLS fitting
Okay great - here we are using two techniques to analyse periodicity: Lomb-Scarg and OLS curve fitting. LS actually >is< OLS at heart I think. OLS stands for 'ordinary least squares' - I think you take a set of curves at a specific period (identified by that LS analysis) and do a least-squares fit onto the data for different phases. It's a targeted analysis at, say, 24H period.

Note that it gives you (importantly!) an r-squared value (how much of the variance is explained as a proportion - ranging from 0 to 1) and a p-value (what's the chance that you would get this fit under the null hypothesis of 'random data')


In [ ]:
%%time
# OLS sinusoid fit at fixed periods (amplitude + lag + significance)
import numpy as np
import pandas as pd

try:
    import statsmodels.api as sm
except ImportError as e:
    raise ImportError(
        "statsmodels is required for the OLS lag analysis. "
        "In Colab run: !pip -q install statsmodels"
    ) from e

try:
    from IPython.display import display
except Exception:
    # Fallback if running outside IPython/Jupyter
    def display(x):
        print(x)

def fit_sinusoid_ols(
    t_hours,
    y,
    period_h: float,
    weights=None,
    fit_intercept: bool = True,
    robust: bool = False,
    cov_type: str = "HC1",
    ):
    """Fit y ~ a*cos(w t) + b*sin(w t) (+ c) via (W)LS and return amplitude + lag.

    Lag interpretation: with model A*cos(w*(t - tau)) + c, the peak occurs at t = tau (mod period).
    """
    t_hours = np.asarray(t_hours, dtype=float)
    y = np.asarray(y, dtype=float)

    # Drop NaNs
    valid_mask = ~np.isnan(y)
    t_hours = t_hours[valid_mask]
    y = y[valid_mask]

    if weights is not None:
        weights = np.asarray(weights, dtype=float)[valid_mask]

    if t_hours.shape != y.shape:
        raise ValueError(f"t_hours and y must have same shape, got {t_hours.shape} vs {y.shape}")
    if period_h <= 0:
        raise ValueError("period_h must be > 0")
    if len(y) == 0:
        raise ValueError("No valid data points left after dropping NaNs.")

    omega = 2.0 * np.pi / period_h
    cos_col = np.cos(omega * t_hours)
    sin_col = np.sin(omega * t_hours)

    if fit_intercept:
        X = pd.DataFrame({"const": 1.0, "cos": cos_col, "sin": sin_col})
    else:
        X = pd.DataFrame({"cos": cos_col, "sin": sin_col})

    y_s = pd.Series(y, name="y")

    if weights is None:
        model = sm.OLS(y_s, X)
    else:
        # WLS expects weights proportional to 1/Var(epsilon); using counts is a pragmatic heuristic.
        # Normalize weights to avoid numerical instability with large counts
        w = weights / np.mean(weights)
        model = sm.WLS(y_s, X, weights=w)

    if robust:
        res = model.fit(cov_type=cov_type)
    else:
        res = model.fit()

    a = float(res.params["cos"])
    b = float(res.params["sin"])
    c = float(res.params["const"]) if fit_intercept else 0.0

    # Joint test H0: cos=0 and sin=0
    try:
        if fit_intercept:
            ftest = res.f_test(np.array([[0, 1, 0], [0, 0, 1]]))
        else:
            ftest = res.f_test(np.array([[1, 0], [0, 1]]))
        p_joint = float(np.asarray(ftest.pvalue).reshape(-1)[0])
        f_joint = float(np.asarray(ftest.fvalue).reshape(-1)[0])
    except Exception:
        p_joint = np.nan
        f_joint = np.nan

    amp = float(np.hypot(a, b))
    # tau in hours; peak at t = tau (mod period)
    tau = float(np.arctan2(b, a) / omega)
    tau_mod = float(tau % period_h)

    return {
        "period_h": float(period_h),
        "freq_per_h": float(1.0 / period_h),
        "n": int(res.nobs),
        "amp": amp,
        "lag_h": tau,
        "lag_h_mod_period": tau_mod,
        "p_joint": p_joint,
        "f_joint": f_joint,
        "r2": float(res.rsquared) if hasattr(res, "rsquared") else np.nan,
        "a_cos": a,
        "b_sin": b,
        "c_const": c,
    }

def fixed_period_lag_table(
    t_hours,
    y,
    periods_h=(12.0, 24.0, 36.0, 48.0, 168.0),
    weights=None,
    alpha: float = 0.05,
    robust: bool = False,
    cov_type: str = "HC1",
    ) -> pd.DataFrame:
    rows = []
    for p in periods_h:
        rows.append(
            fit_sinusoid_ols(
                t_hours=t_hours,
                y=y,
                period_h=float(p),
                weights=weights,
                fit_intercept=True,
                robust=robust,
                cov_type=cov_type,
            )
        )
    out = pd.DataFrame(rows).sort_values("period_h").reset_index(drop=True)
    out["significant"] = out["p_joint"] < alpha
    return out

# --- Run on the current hourly target series ---
if "hourly" not in globals() or hourly is None or len(hourly) == 0:
    raise RuntimeError("Expected `hourly` DataFrame to exist and be non-empty.")

t_for_fit = hourly["hour_idx"].to_numpy(dtype=float, copy=True)
t_for_fit -= t_for_fit.min()
y_for_fit = hourly["target_mean"].to_numpy(dtype=float, copy=True)

# Fallback to standard OLS without weights to prevent WLS covariance matrix issues
w_for_fit = None

periods_to_check = (12.0, 24.0, 36.0, 48.0, 24.0 * 7.0)
tbl = fixed_period_lag_table(
    t_hours=t_for_fit,
    y=y_for_fit,
    periods_h=periods_to_check,
    weights=w_for_fit,
    alpha=0.05,
    robust=False,
    cov_type="HC1",
 )

display_cols = ["period_h", "amp", "lag_h_mod_period", "p_joint", "r2", "n", "significant"]
print("OLS sinusoid fits at fixed periods (lag is hours-of-cycle where the fitted cosine peaks):")
display(tbl[display_cols])

sig = tbl[tbl["significant"]].copy()
if len(sig) == 0:
    print("\nNo periods significant at alpha=0.05 (joint test cos=sin=0).")
else:
    print("\nSignificant periods (alpha=0.05):")
    display(sig[display_cols])


# Standalone Win-Rate Rhythm

Before adding win/loss to PCA, analyze win rate by itself. This keeps the outcome question separate from the performance-behavior components.


In [ ]:
%%time
# Standalone win-rate analysis on the same filtered hourly window.
win_rate_query = """
WITH game_wins AS (
    SELECT
        CAST(FLOOR(CAST(TIMESTAMP AS DOUBLE) / 3600000.0) AS BIGINT) AS hour_idx,
        CASE
            WHEN LOWER(WINLOSE) = 'true' THEN 1.0
            WHEN LOWER(WINLOSE) = 'false' THEN 0.0
            ELSE NULL
        END AS win_flag
    FROM riotData
    WHERE PLATFORMID = ?
      AND TIMESTAMP IS NOT NULL
      AND WINLOSE IS NOT NULL
      AND WINLOSE <> ''
)
SELECT
    hour_idx,
    AVG(win_flag) AS win_rate,
    COUNT(win_flag) AS n_win_records
FROM game_wins
WHERE win_flag IS NOT NULL
GROUP BY hour_idx
ORDER BY hour_idx;
"""

conn = connect_db(db_path)
hourly_win_raw = pd.read_sql_query(win_rate_query, conn, params=[platform])
conn.close()

analysis_hours = hourly[['hour_idx']].drop_duplicates().copy()
hourly_win = analysis_hours.merge(hourly_win_raw, on='hour_idx', how='inner')

if hourly_win.empty:
    raise RuntimeError(f"No win-rate rows found for PLATFORMID={platform} in the filtered analysis window.")

print(f"Win-rate rows in filtered window: {len(hourly_win)}")
print(f"Mean hourly win rate: {hourly_win['win_rate'].mean():.4f}")

win_t_hours = hourly_win['hour_idx'].to_numpy(dtype=float, copy=True)
win_t_hours -= win_t_hours.min()
win_y = hourly_win['win_rate'].to_numpy(dtype=float, copy=True)
win_y_demeaned = win_y - np.mean(win_y)

frequency = np.linspace(1.0 / max_period_h, 1.0 / min_period_h, n_freq)
period = 1.0 / frequency
win_power = LombScargle(win_t_hours, win_y_demeaned).power(frequency)
win_best_idx = int(np.argmax(win_power))
win_best_period = float(period[win_best_idx])
win_power_24 = float(win_power[int(np.argmin(np.abs(period - 24.0)))])

print(f"Win-rate best period: {win_best_period:.3f} h")
print(f"Win-rate power near 24 h: {win_power_24:.5f}")

win_fit_tbl = fixed_period_lag_table(
    t_hours=win_t_hours,
    y=win_y,
    periods_h=(12.0, 24.0, 36.0, 48.0, 24.0 * 7.0),
    weights=hourly_win['n_win_records'].to_numpy(dtype=float),
    alpha=0.05,
    robust=True,
    cov_type="HC1",
)
print("OLS sinusoid fits for hourly win rate:")
display(win_fit_tbl[["period_h", "amp", "lag_h_mod_period", "p_joint", "r2", "n", "significant"]])

# Local-hour folded win rate, weighted by sampled player-match records per hour bin.
offset_hours = platform_utc_offset_hours(platform)
hourly_win['local_hour'] = np.floor((hourly_win['hour_idx'] + offset_hours) % 24.0).astype(int)
local_win_rows = []
for local_hour, group in hourly_win.groupby('local_hour'):
    local_win_rows.append({
        'local_hour': local_hour,
        'win_rate': np.average(group['win_rate'], weights=group['n_win_records']),
        'n_win_records': group['n_win_records'].sum(),
    })
local_win = pd.DataFrame(local_win_rows).set_index('local_hour').reindex(range(24)).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={"width_ratios": [1.25, 1.0, 1.0]})

ax = axes[0]
ax.plot(win_t_hours, win_y_demeaned, color=COLORS['primary'], linewidth=1.0, alpha=0.9)
ax.axhline(0.0, color=COLORS['muted'], linestyle='--', linewidth=1.0)
ax.set_title(f"Hourly Win Rate Over Time ({platform})")
ax.set_xlabel("Hours Since Start")
ax.set_ylabel("Win rate (demeaned)")
style_axes(ax)

ax = axes[1]
ax.plot(period, win_power, color=COLORS['secondary'], linewidth=2.0)
ax.axvline(24.0, color=COLORS['accent'], linestyle='--', linewidth=1.3, label='24 h')
ax.axvline(win_best_period, color=COLORS['ink'], linestyle=':', linewidth=1.3, label=f'Peak {win_best_period:.1f} h')
ax.set_xlim(0, max_period_h)
ax.set_title(f"Win-Rate Periodogram ({platform})")
ax.set_xlabel("Period (hours)")
ax.set_ylabel("Power")
ax.legend(frameon=False)
style_axes(ax)

ax = axes[2]
ax.plot(local_win['local_hour'], local_win['win_rate'], color=COLORS['accent'], marker='o', linewidth=2.0)
ax.axhline(hourly_win['win_rate'].mean(), color=COLORS['muted'], linestyle='--', linewidth=1.1, label='Mean')
ax.set_title(f"Win Rate by Local Hour ({platform})")
ax.set_xlabel("Local hour")
ax.set_ylabel("Win rate")
ax.set_xticks(np.arange(0, 24, 2))
ax.legend(frameon=False)
style_axes(ax, grid_axis='y')

fig.suptitle(f"Standalone Win-Rate Rhythm: PLATFORMID={platform}", fontsize=13.5, fontweight='bold', color=COLORS['ink'])
plt.tight_layout()
plt.show()


# A note on rsquared and sig

We have a lot of datapoints. That is common in 'big data' - and so we have a lot of 'power' - an ability to see small effects. The effects we see above have enormously small p-values (so they are very significant) but also quite low rsquared vals (they explain only a small fraction of the variance). So make of that what you will...

# PCA analysis

So far we've focused on a single variable (GOLD per time). That is probably a reasonable proxy for some sort of performance but it's not the whole story. Also different players collect gold at different rates depending on their position - and different games might be chaotic or not - and that in turn might affect GOLD harvesting. We can generate a more robust performance metric by looking at a principal components analysis of all the variables. We then use that component later....

In [ ]:
%%time
# Load multi-metric data for PCA analysis
metric_map = {
    "NEUTRALCREEP": "neutralcreep_mean",
    "ENEMYCREEP": "enemycreep_mean",
    "GOLD": "gold_mean",
    "DAMDEALT": "damdealt_mean",
    "TIMEDEAD": "timedead_mean",
    "TIMEPLAYED": "timeplayed_mean",
    "KILLS": "kills_mean",
    "DEATHS": "deaths_mean",
    "ASSISTS": "assists_mean",
}

select_metrics = ",\n  ".join([f"{alias}" for alias in metric_map.values()])

gold_query = f"""
SELECT
  hour_idx,
  {select_metrics},
  n
FROM hourly_agg
WHERE platformid = ?
ORDER BY hour_idx;
"""

conn = connect_db(db_path)
hourly_gold = pd.read_sql_query(gold_query, conn, params=[platform])
conn.close()

# Apply same filtering as before
min_hour = hourly_gold['hour_idx'].min()
max_hour = hourly_gold['hour_idx'].max()
duration_hours = max_hour - min_hour

# Note the filtering by MAX_HOUR_LIMIT!
if duration_hours > MAX_HOUR_LIMIT:
    cutoff_idx = min_hour + MAX_HOUR_LIMIT
    hourly_gold = hourly_gold[hourly_gold['hour_idx'] <= cutoff_idx]

# IQR filtering on hour_idx
Q1 = hourly_gold['hour_idx'].quantile(0.25)
Q3 = hourly_gold['hour_idx'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 3 * IQR
hourly_gold = hourly_gold[(hourly_gold['hour_idx'] >= lower_bound) &
                           (hourly_gold['hour_idx'] <= upper_bound)]

# Build PCA inputs as rates normalized by time played.
# TIMEPLAYED itself is not included because timeplayed / timeplayed is constant.
# Riot TIMEPLAYED is in seconds, so divide by 60 to get minutes.
timeplayed_minutes = (hourly_gold['timeplayed_mean'] / 60.0).where(hourly_gold['timeplayed_mean'] > 0)

pca_rate_map = {
    'neutralcreep_per_min': 'neutralcreep_mean',
    'enemycreep_per_min': 'enemycreep_mean',
    'gold_per_min': 'gold_mean',
    'damdealt_per_min': 'damdealt_mean',
    'kills_per_min': 'kills_mean',
    'deaths_per_min': 'deaths_mean',
    'assists_per_min': 'assists_mean',
}

for rate_col, source_col in pca_rate_map.items():
    hourly_gold[rate_col] = hourly_gold[source_col] / timeplayed_minutes

hourly_gold['timedead_fraction'] = hourly_gold['timedead_mean'] / hourly_gold['timeplayed_mean'].where(hourly_gold['timeplayed_mean'] > 0)

# Keep this for the later single-metric comparison plot; it is not a PCA input.
hourly_gold['gold_per_sec'] = hourly_gold['gold_mean'] / hourly_gold['timeplayed_mean'].where(hourly_gold['timeplayed_mean'] > 0)

numeric_cols = list(pca_rate_map.keys()) + ['timedead_fraction']

print(f"Multi-metric data: {len(hourly_gold)} hourly bins, {len(numeric_cols)} PCA features")
print(f"PCA input columns: {numeric_cols}")
print(f"GOLD/MIN range: {hourly_gold['gold_per_min'].min():.3f} - {hourly_gold['gold_per_min'].max():.3f}")
print(f"DAMAGE/MIN range: {hourly_gold['damdealt_per_min'].min():.3f} - {hourly_gold['damdealt_per_min'].max():.3f}")
print(f"TIMEDEAD fraction range: {hourly_gold['timedead_fraction'].min():.3f} - {hourly_gold['timedead_fraction'].max():.3f}")


In [ ]:
%%time
# Outlier rejection: remove large peaks from the time-normalized PCA inputs.
print("Outlier rejection by PCA input metric:")
print("-" * 60)

initial_rows = len(hourly_gold)
rows_per_metric = {}

for col in numeric_cols:
    Q1 = hourly_gold[col].quantile(0.25)
    Q3 = hourly_gold[col].quantile(0.75)
    IQR = Q3 - Q1

    # Define bounds: Q1 - 1.5*IQR (lower), Q3 + 1.5*IQR (upper)
    # This catches outliers on both sides
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    before = len(hourly_gold)
    hourly_gold = hourly_gold[(hourly_gold[col] >= lower) & (hourly_gold[col] <= upper)]
    after = len(hourly_gold)
    removed = before - after

    rows_per_metric[col] = {'before': before, 'after': after, 'removed': removed}

    if removed > 0:
        pct = 100.0 * removed / before
        print(f"  {col:25s}: {before:5d} → {after:5d}  (removed {removed:4d}, {pct:5.2f}%)")

total_removed = initial_rows - len(hourly_gold)
pct_total = 100.0 * total_removed / initial_rows
print("-" * 60)
print(f"Total: {initial_rows} → {len(hourly_gold)} rows (removed {total_removed}, {pct_total:.2f}%)")

# Verify we still have data
if hourly_gold.empty:
    raise RuntimeError("All rows were filtered out!")

# Recompute true gold per second for the later single-metric comparison plot.
hourly_gold['gold_per_sec'] = hourly_gold['gold_mean'] / hourly_gold['timeplayed_mean'].where(hourly_gold['timeplayed_mean'] > 0)


In [ ]:
%%time
# PCA factors from outlier-cleaned, time-normalized performance fields
features = hourly_gold[numeric_cols].dropna().copy()
X = features.to_numpy(dtype=float)

# Standardize features
X_mean = X.mean(axis=0)
X_std = X.std(axis=0, ddof=0)
X_std[X_std == 0] = 1.0
X_z = (X - X_mean) / X_std

# PCA via SVD
U, S, Vt = np.linalg.svd(X_z, full_matrices=False)
explained = (S ** 2) / np.sum(S ** 2)

good_pca_cols = ['gold_per_min', 'damdealt_per_min', 'kills_per_min', 'assists_per_min']

# PCA sign is arbitrary. Orient all components so the average good-rate loading is positive.
pca_component_signs = np.ones(Vt.shape[0])
for component_index in range(Vt.shape[0]):
    component_loadings = pd.Series(Vt[component_index].copy(), index=numeric_cols)
    if component_loadings[good_pca_cols].mean() < 0:
        Vt[component_index, :] *= -1.0
        pca_component_signs[component_index] = -1.0

# Keep the first two PCA components for downstream player projections.
pca_components = {
    'PC1': 'perf_factor_pc1',
    'PC2': 'perf_factor_pc2',
}
pca_component_loadings = {}

# Keep hour_idx for time-series analysis and timeplayed_mean for reference.
hourly_gold_factor = hourly_gold.loc[features.index, ['hour_idx', 'timeplayed_mean']].copy()

for component_index, (component_label, score_col) in enumerate(pca_components.items()):
    component_scores = U[:, component_index] * S[component_index] * pca_component_signs[component_index]
    component_loadings = pd.Series(Vt[component_index].copy(), index=numeric_cols)

    hourly_gold_factor[score_col] = component_scores
    pca_component_loadings[component_label] = component_loadings

# Backward-compatible alias for earlier PC1-only cells.
hourly_gold_factor['perf_factor'] = hourly_gold_factor['perf_factor_pc1']
loadings = pca_component_loadings['PC1']

print("Explained variance ratio (outliers removed):")
print("  " + ", ".join([f"PC{i+1}: {explained[i]:.3f}" for i in range(min(5, len(explained)))]))
for component_label, component_loadings in pca_component_loadings.items():
    print(f"\n{component_label} loadings (oriented so average gold/damage/kills/assists loading is positive):")
    print(component_loadings.sort_values(ascending=False))

# Multi-panel visual summary for the first two PCA factors
fig, axes = plt.subplots(
    len(pca_components),
    2,
    figsize=(14, 4.0 * len(pca_components)),
    gridspec_kw={"width_ratios": [1.35, 1.0]},
)
axes = np.atleast_2d(axes)

t_hours_factor = hourly_gold_factor['hour_idx'].to_numpy(dtype=float, copy=True)
t_hours_factor -= t_hours_factor.min()
component_colors = {'PC1': COLORS["primary"], 'PC2': COLORS["accent"]}
hist_colors = {'PC1': COLORS["secondary"], 'PC2': COLORS["highlight"]}

for row_idx, (component_label, score_col) in enumerate(pca_components.items()):
    ax = axes[row_idx, 0]
    ax.plot(
        t_hours_factor,
        hourly_gold_factor[score_col],
        linewidth=0.9,
        color=component_colors.get(component_label, COLORS["primary"]),
        alpha=0.85,
    )
    ax.axhline(0.0, color=COLORS["muted"], linewidth=1.0, linestyle='--', alpha=0.8)
    ax.set_title(f"{component_label} Over Time ({platform})")
    ax.set_xlabel("Hours Since Start")
    ax.set_ylabel(f"{component_label} Score (per-minute inputs)")
    style_axes(ax)

    ax = axes[row_idx, 1]
    ax.hist(
        hourly_gold_factor[score_col],
        bins=55,
        color=hist_colors.get(component_label, COLORS["secondary"]),
        alpha=0.85,
        edgecolor="white",
    )
    ax.axvline(
        hourly_gold_factor[score_col].mean(),
        color=COLORS["accent"] if component_label == 'PC1' else COLORS["primary"],
        linewidth=2.0,
        linestyle='--',
        label='Mean',
    )
    ax.set_title(f"{component_label} Distribution ({platform})")
    ax.set_xlabel(f"{component_label} Score")
    ax.set_ylabel("Frequency")
    ax.legend(frameon=False)
    style_axes(ax, grid_axis='y')

fig.suptitle(f"PCA Performance Factors Snapshot (per-minute inputs): PLATFORMID={platform}", fontsize=13, fontweight='bold', color=COLORS["ink"])
plt.tight_layout()
plt.show()


### Scree Plot and Component Loadings

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Scree + cumulative explained variance in one panel
x = np.arange(1, len(explained) + 1)
cum_explained = np.cumsum(explained)

fig, ax = plt.subplots(figsize=(9.5, 5.0))
ax.bar(x, explained, color=COLORS["secondary"], alpha=0.82, label='Per-component variance')
ax.plot(x, explained, color=COLORS["primary"], linewidth=2.0, marker='o', markersize=4)
ax.plot(x, cum_explained, color=COLORS["accent"], linewidth=2.2, marker='s', markersize=3.5, label='Cumulative variance')

ax.set_title(f'Scree Plot With Cumulative Variance, PLATFORMID={platform}')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance Ratio')
ax.set_xticks(x)
ax.set_ylim(0, 1.02)
style_axes(ax)
ax.legend(frameon=False, loc='upper right')

plt.tight_layout()
plt.show()

print("Loadings for the first 3 Principal Components:")
for i in range(min(3, len(explained))):
    print(f"\nPC{i+1} loadings:")
    loadings_pc = pd.Series(Vt[i], index=numeric_cols)
    display(loadings_pc.sort_values(ascending=False))

# Success-Aware PCA

Now add hourly win rate to the PCA as a sensitivity analysis. This deliberately creates a second PCA, so the original performance-behavior PCA remains interpretable on its own.


In [ ]:
%%time
# Success-aware PCA: same time-normalized performance inputs plus hourly win rate.
if 'hourly_win' not in globals() or hourly_win.empty:
    raise RuntimeError("Run the standalone win-rate analysis before success-aware PCA.")

success_pca_input = hourly_gold.merge(
    hourly_win[['hour_idx', 'win_rate', 'n_win_records']],
    on='hour_idx',
    how='inner',
)

success_numeric_cols = numeric_cols + ['win_rate']
success_features = success_pca_input[success_numeric_cols].dropna().copy()
X_success = success_features.to_numpy(dtype=float)

success_X_mean = X_success.mean(axis=0)
success_X_std = X_success.std(axis=0, ddof=0)
success_X_std[success_X_std == 0] = 1.0
X_success_z = (X_success - success_X_mean) / success_X_std

U_success, S_success, Vt_success = np.linalg.svd(X_success_z, full_matrices=False)
success_explained = (S_success ** 2) / np.sum(S_success ** 2)

success_good_cols = good_pca_cols + ['win_rate']
success_component_signs = np.ones(Vt_success.shape[0])
for component_index in range(Vt_success.shape[0]):
    loadings_component = pd.Series(Vt_success[component_index].copy(), index=success_numeric_cols)
    if loadings_component[success_good_cols].mean() < 0:
        Vt_success[component_index, :] *= -1.0
        success_component_signs[component_index] = -1.0

success_pca_components = {
    'Success PC1': 'success_factor_pc1',
    'Success PC2': 'success_factor_pc2',
    'Success PC3': 'success_factor_pc3',
}
success_pca_loadings = {}
hourly_success_factor = success_pca_input.loc[success_features.index, ['hour_idx', 'win_rate', 'n_win_records']].copy()

for component_index, (component_label, score_col) in enumerate(success_pca_components.items()):
    scores = U_success[:, component_index] * S_success[component_index] * success_component_signs[component_index]
    loadings_component = pd.Series(Vt_success[component_index].copy(), index=success_numeric_cols)
    hourly_success_factor[score_col] = scores
    success_pca_loadings[component_label] = loadings_component

print("Success-aware PCA explained variance ratio:")
print("  " + ", ".join([f"PC{i+1}: {success_explained[i]:.3f}" for i in range(min(5, len(success_explained)))]))
for component_label, loadings_component in success_pca_loadings.items():
    print(f"\n{component_label} loadings (including win_rate):")
    print(loadings_component.sort_values(ascending=False))

fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), sharex=False)
for ax, (component_label, loadings_component) in zip(axes, success_pca_loadings.items()):
    ordered = loadings_component.sort_values()
    colors = [COLORS['accent'] if value >= 0 else COLORS['primary'] for value in ordered]
    ax.barh(ordered.index, ordered.values, color=colors, alpha=0.88)
    ax.axvline(0.0, color=COLORS['ink'], linewidth=1.0)
    ax.set_title(f"{component_label} Loadings")
    ax.set_xlabel("Loading")
    style_axes(ax, grid_axis='x')

fig.suptitle(f"Success-Aware PCA Loadings: PLATFORMID={platform}", fontsize=13.5, fontweight='bold', color=COLORS['ink'])
plt.tight_layout()
plt.show()

# Periodograms and fixed-period OLS checks for success-aware PCs.
success_t_hours = hourly_success_factor['hour_idx'].to_numpy(dtype=float, copy=True)
success_t_hours -= success_t_hours.min()
frequency = np.linspace(1.0 / max_period_h, 1.0 / min_period_h, n_freq)
period = 1.0 / frequency

success_pca_periodogram_results = {}
success_pca_fit_tables = {}
fig, axes = plt.subplots(1, len(success_pca_components), figsize=(6.2 * len(success_pca_components), 4.8), sharey=True)
axes = np.atleast_1d(axes)

for ax, (component_label, score_col) in zip(axes, success_pca_components.items()):
    y_success = hourly_success_factor[score_col].to_numpy(dtype=float, copy=True)
    y_success -= np.mean(y_success)
    power_success = LombScargle(success_t_hours, y_success).power(frequency)
    best_idx = int(np.argmax(power_success))
    best_period_success = float(period[best_idx])
    power_24_success = float(power_success[int(np.argmin(np.abs(period - 24.0)))])

    success_pca_periodogram_results[component_label] = {
        'score_col': score_col,
        'period': period,
        'power': power_success,
        'best_period': best_period_success,
        'power_24': power_24_success,
    }

    fit_table = fixed_period_lag_table(
        t_hours=success_t_hours,
        y=hourly_success_factor[score_col].to_numpy(dtype=float, copy=True),
        periods_h=(12.0, 24.0, 36.0, 48.0, 24.0 * 7.0),
        weights=None,
        alpha=0.05,
        robust=True,
        cov_type="HC1",
    )
    fit_table.insert(0, 'component', component_label)
    success_pca_fit_tables[component_label] = fit_table

    print(f"{component_label}: best period = {best_period_success:.3f} h, power near 24 h = {power_24_success:.5f}")
    display(fit_table[['component', 'period_h', 'amp', 'lag_h_mod_period', 'p_joint', 'r2', 'n', 'significant']])

    ax.plot(period, power_success, color=COLORS['secondary'], linewidth=2.0)
    ax.axvline(24.0, color=COLORS['accent'], linestyle='--', linewidth=1.2, label='24 h')
    ax.axvline(best_period_success, color=COLORS['ink'], linestyle=':', linewidth=1.2, label=f'Peak {best_period_success:.1f} h')
    ax.set_xlim(0, max_period_h)
    ax.set_title(f"{component_label} Periodogram")
    ax.set_xlabel("Period (hours)")
    ax.legend(frameon=False)
    style_axes(ax, grid_axis='y')

axes[0].set_ylabel("Power")
fig.suptitle(f"Success-Aware PCA Periodograms: PLATFORMID={platform}", fontsize=13.5, fontweight='bold', color=COLORS['ink'])
plt.tight_layout()
plt.show()

success_pca_fit_summary = pd.concat(success_pca_fit_tables.values(), ignore_index=True)


# Periodogram of PCA
We have extracted a PCA component from time-normalized performance data, using per-minute rates such as GOLD/MIN and DAMAGE/MIN rather than raw game totals.

Now we can compute a periodogram on the data using the LS method - we're hoping it will be even cleaner than the individual metrics we had before.

In [ ]:
%%time
# Periodograms of PCA factors built from per-minute inputs
# Do not divide PCA scores by time played again; their inputs are already time-normalized.
factor_df = hourly_gold_factor.copy()

if 'pca_components' not in globals():
    pca_components = {'PC1': 'perf_factor'}

pca_components_to_analyze = {
    component_label: score_col
    for component_label, score_col in pca_components.items()
    if score_col in factor_df.columns
}
if not pca_components_to_analyze:
    raise RuntimeError("No PCA component score columns found in factor_df.")

# Backward-compatible alias for PC1-only downstream code.
pc1_score_col = pca_components_to_analyze.get('PC1', next(iter(pca_components_to_analyze.values())))
factor_df['perf_factor_for_periodogram'] = factor_df[pc1_score_col]

# Lomb-Scargle periodogram setup
min_freq = 1.0 / max_period_h
max_freq = 1.0 / min_period_h
frequency = np.linspace(min_freq, max_freq, n_freq)
period = 1.0 / frequency

pca_periodogram_results = {}
fig, axes = plt.subplots(1, len(pca_components_to_analyze), figsize=(7.0 * len(pca_components_to_analyze), 5), sharey=True)
axes = np.atleast_1d(axes)

for ax, (component_label, score_col) in zip(axes, pca_components_to_analyze.items()):
    ft_hours = factor_df['hour_idx'].to_numpy(dtype=float, copy=True)
    ft_hours -= ft_hours.min()

    fy = factor_df[score_col].to_numpy(dtype=float, copy=True)
    fy -= np.mean(fy)

    ls = LombScargle(ft_hours, fy)
    power_component = ls.power(frequency)

    best_idx = int(np.argmax(power_component))
    best_period_component = float(period[best_idx])
    power_24_component = float(power_component[int(np.argmin(np.abs(period - 24.0)))])

    pca_periodogram_results[component_label] = {
        'score_col': score_col,
        'period': period,
        'power': power_component,
        'best_period': best_period_component,
        'power_24': power_24_component,
    }

    print(f"{component_label} PCA factor (per-minute inputs, outliers removed) best period: {best_period_component:.3f} h")
    print(f"{component_label} PCA factor (per-minute inputs, outliers removed) power at 24h: {power_24_component:.5f}")

    ax.bar(period, power_component, width=0.2, color=COLORS["secondary"], alpha=0.7)
    ax.axvline(24.0, linestyle="--", linewidth=1, color=COLORS["accent"], alpha=0.6, label='24h')
    if max_period_h >= 168:
        ax.axvline(168.0, linestyle="--", linewidth=1, color=COLORS["primary"], alpha=0.5, label='168h')
    ax.set_xlim(0, max_period_h)
    ax.set_title(f"Periodogram: {component_label}, PLATFORMID={platform}")
    ax.set_xlabel("Period (hours)")
    ax.set_ylabel("Power")
    ax.legend(loc='upper right', frameon=False)
    style_axes(ax, grid_axis='y')

# Backward-compatible variables for PC1-only cells/notes.
pc1_periodogram = pca_periodogram_results.get('PC1', next(iter(pca_periodogram_results.values())))
power = pc1_periodogram['power']
best_period = pc1_periodogram['best_period']
power_24 = pc1_periodogram['power_24']

fig.suptitle(f"PCA Factor Periodograms From Per-Minute Inputs: PLATFORMID={platform}", fontsize=13, fontweight='bold', color=COLORS["ink"])
plt.tight_layout()
plt.show()

In [ ]:
%%time
# Apply per-metric filtering and visualization
metrics = {
    'GOLD': 'gold_mean',
    'TIMEPLAYED': 'timeplayed_mean',
    'GOLD/SEC': 'gold_per_sec'
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
results = {}

for idx, (metric_name, col_name) in enumerate(metrics.items()):
    # IQR-based filtering
    Q1 = hourly_gold[col_name].quantile(0.25)
    Q3 = hourly_gold[col_name].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 3 * IQR

    filtered = hourly_gold[(hourly_gold[col_name] >= lower_bound) &
                           (hourly_gold[col_name] <= upper_bound)].copy()

    print(f"{metric_name}: {len(hourly_gold)} → {len(filtered)} (removed {len(hourly_gold) - len(filtered)} outliers)")

    results[metric_name] = {
        'data': filtered,
        'col': col_name,
        'mean': filtered[col_name].mean(),
        'std': filtered[col_name].std()
    }

    # Histogram
    styled_hist(axes[idx], filtered[col_name], bins=50, variant="secondary")
    axes[idx].set_title(f'{metric_name}, PLATFORMID={platform}')
    axes[idx].set_xlabel(metric_name)
    axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
%%time
# Lomb-Scargle periodogram for all three metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (metric_name, metric_info) in enumerate(results.items()):
    filtered_data = metric_info['data']
    col_name = metric_info['col']

    # Prepare time and value arrays
    t_hours_gold = filtered_data['hour_idx'].to_numpy(dtype=float, copy=True)
    t_hours_gold -= t_hours_gold.min()

    y_gold = filtered_data[col_name].to_numpy(dtype=float, copy=True)
    y_gold -= np.mean(y_gold)

    # Calculate periodogram
    min_freq = 1.0 / max_period_h
    max_freq = 1.0 / min_period_h
    frequency = np.linspace(min_freq, max_freq, n_freq)

    ls = LombScargle(t_hours_gold, y_gold)
    power = ls.power(frequency)
    period = 1.0 / frequency

    best_idx = int(np.argmax(power))
    best_period = float(period[best_idx])
    power_24 = float(power[int(np.argmin(np.abs(period - 24.0)))])

    print(f"\n{metric_name}:")
    print(f"  Best period: {best_period:.3f} h")
    print(f"  Power at 24h: {power_24:.5f}")

    # Plot
    ax = axes[idx]
    ax.bar(period, power, width=0.2, color='steelblue', alpha=0.7)
    ax.axvline(24.0, linestyle="--", linewidth=1, color='r', alpha=0.5, label='24h')
    if max_period_h >= 168:
        ax.axvline(168.0, linestyle="--", linewidth=1, color='g', alpha=0.5, label='168h')
    ax.set_xlim(0, max_period_h)
    ax.set_xlabel('Period (hours)')
    ax.set_ylabel('Power')
    ax.set_title(f'Periodogram: {metric_name}, PLATFORMID={platform}')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Time-series plots for all three metrics
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for idx, (metric_name, metric_info) in enumerate(results.items()):
    filtered_data = metric_info['data']
    col_name = metric_info['col']

    t_hours_plot = filtered_data['hour_idx'].to_numpy(dtype=float, copy=True)
    t_hours_plot -= t_hours_plot.min()
    y_plot = filtered_data[col_name].to_numpy(dtype=float, copy=True)
    y_plot_demeaned = y_plot - np.mean(y_plot)

    ax = axes[idx]
    ax.plot(t_hours_plot, y_plot_demeaned, linewidth=0.8, color='steelblue')
    ax.set_title(f'Hourly mean {metric_name} (demeaned), PLATFORMID={platform}')
    ax.set_ylabel(f'{metric_name}')
    ax.grid(alpha=0.3)

    if idx == 2:
        ax.set_xlabel('Hours since start')

plt.tight_layout()
plt.show()


# PCA sinusoids?
Can we look at single sinusoids of the PCA

In [ ]:
%%time
# OLS sinusoid fits for PCA factors from per-minute inputs

if "factor_df" not in globals() or factor_df is None or len(factor_df) == 0:
    raise RuntimeError("Expected `factor_df` DataFrame to exist and be non-empty.")

if 'pca_components_to_analyze' not in globals():
    pca_components_to_analyze = {
        component_label: score_col
        for component_label, score_col in globals().get('pca_components', {'PC1': 'perf_factor'}).items()
        if score_col in factor_df.columns
    }

periods_to_check_pca = (12.0, 24.0, 36.0, 48.0, 24.0 * 7.0)
display_cols_pca = ["period_h", "amp", "lag_h_mod_period", "p_joint", "r2", "n", "significant"]
pca_fit_tables = {}

for component_label, score_col in pca_components_to_analyze.items():
    t_for_fit_pca = factor_df["hour_idx"].to_numpy(dtype=float, copy=True)
    t_for_fit_pca -= t_for_fit_pca.min()
    y_for_fit_pca = factor_df[score_col].to_numpy(dtype=float, copy=True)

    w_for_fit_pca = None  # No direct 'n' column for weighting the PCA factor.

    tbl_component = fixed_period_lag_table(
        t_hours=t_for_fit_pca,
        y=y_for_fit_pca,
        periods_h=periods_to_check_pca,
        weights=w_for_fit_pca,
        alpha=0.05,
        robust=True,
        cov_type="HC1",
    )
    tbl_component.insert(0, 'component', component_label)
    pca_fit_tables[component_label] = tbl_component

    print(f"OLS sinusoid fits at fixed periods for {component_label} PCA factor from per-minute inputs:")
    display(tbl_component[['component'] + display_cols_pca])

    sig_component = tbl_component[tbl_component["significant"]].copy()
    if len(sig_component) == 0:
        print(f"\nNo {component_label} periods significant at alpha=0.05 (joint test cos=sin=0).")
    else:
        print(f"\nSignificant periods (alpha=0.05) for {component_label} PCA factor:")
        display(sig_component[['component'] + display_cols_pca])

pca_fit_summary = pd.concat(pca_fit_tables.values(), ignore_index=True)

# Backward-compatible aliases for PC1-only downstream code.
tbl_pca = pca_fit_tables.get('PC1', next(iter(pca_fit_tables.values())))
tbl_pca_pc2 = pca_fit_tables.get('PC2')
sig_pca = tbl_pca[tbl_pca["significant"]].copy()

In [ ]:
%%time
# Plot the 24-hour folded PCA factors and fitted sinusoids
import matplotlib.pyplot as plt
import numpy as np

if 'pca_fit_tables' not in globals() or not pca_fit_tables:
    raise RuntimeError("Expected `pca_fit_tables` from the PCA OLS fit cell.")

components_for_plot = {
    component_label: score_col
    for component_label, score_col in pca_components_to_analyze.items()
    if component_label in pca_fit_tables
}

n_components = len(components_for_plot)
t_cycle = np.linspace(0, 24, 240)
omega = 2.0 * np.pi / 24.0
bins = np.linspace(0, 24, 25)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

fig, axes = plt.subplots(
    n_components,
    2,
    figsize=(14.5, 4.8 * n_components),
    gridspec_kw={"width_ratios": [1.35, 1.0]},
)
axes = np.atleast_2d(axes)

for row_idx, (component_label, score_col) in enumerate(components_for_plot.items()):
    row_24 = pca_fit_tables[component_label][pca_fit_tables[component_label]['period_h'] == 24.0].iloc[0]
    a_cos = row_24['a_cos']
    b_sin = row_24['b_sin']
    c_const = row_24['c_const']
    lag = row_24['lag_h_mod_period']

    y_fit = a_cos * np.cos(omega * t_cycle) + b_sin * np.sin(omega * t_cycle) + c_const

    t_empirical = factor_df['hour_idx'].to_numpy(dtype=float)
    t_empirical -= t_empirical.min()
    t_empirical_folded = t_empirical % 24.0
    y_empirical = factor_df[score_col].to_numpy(dtype=float)

    counts, _ = np.histogram(t_empirical_folded, bins=bins)
    weights, _ = np.histogram(t_empirical_folded, bins=bins, weights=y_empirical)
    bin_means = np.divide(weights, counts, out=np.zeros_like(weights), where=counts != 0)

    ax = axes[row_idx, 0]
    ax.scatter(
        t_empirical_folded,
        y_empirical,
        alpha=0.16,
        color=COLORS["secondary"],
        s=16,
        label='Hourly points',
    )
    ax.plot(bin_centers, bin_means, color=COLORS["ink"], marker='o', markersize=5, linewidth=2, label='Binned mean')
    ax.plot(t_cycle, y_fit, color=COLORS["accent"], linewidth=3, label=f'24 h fit (peak ~{lag:.1f} h)')
    ax.set_title(f"Folded {component_label} Signal vs 24 h Fit ({platform})")
    ax.set_xlabel("Hour in 24 h Cycle")
    ax.set_ylabel(f"{component_label} Factor (per-minute inputs)")
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 25, 2))
    style_axes(ax)
    ax.legend(frameon=False, loc='upper right')

    ax = axes[row_idx, 1]
    y_fit_at_bins = a_cos * np.cos(omega * bin_centers) + b_sin * np.sin(omega * bin_centers) + c_const
    residuals = bin_means - y_fit_at_bins
    bar_colors = [COLORS["accent"] if r >= 0 else COLORS["primary"] for r in residuals]
    ax.bar(bin_centers, residuals, width=0.82, color=bar_colors, alpha=0.85)
    ax.axhline(0, color=COLORS["ink"], linewidth=1.2)
    ax.set_title(f"{component_label} Binned Residuals (Mean - Fit), PLATFORMID={platform}")
    ax.set_xlabel("Hour in 24 h Cycle")
    ax.set_ylabel("Residual")
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 25, 3))
    style_axes(ax, grid_axis='y')

fig.suptitle(f"24 h Circadian Fit Diagnostics: PLATFORMID={platform}", fontsize=13.5, fontweight='bold', color=COLORS["ink"])
plt.tight_layout()
plt.show()

Finally: We see clear cycles in the binned data - particularly at 24Hr period. This could be due to two things:

1: Individual people play throughout the day and they have individual circadian rhythms (peaking early in the morning)
2: Different people play at different times of day - the good/crazy players are logging in early in the day.


First: Plot a histogram showing actual game start times - are there differences in the volumes of games over the day?

In [ ]:
%%time
import matplotlib.pyplot as plt
import numpy as np

# Use the shared fixed UTC offset helper for platform local time.
offset_hours = platform_utc_offset_hours(platform)
print(f"Applying UTC offset of {offset_hours:+d} hours for {platform}")

# Calculate local hour by adding the offset before taking modulo 24
local_hour = np.floor((hourly_gold['hour_idx'] + offset_hours) % 24.0).astype(int)
counts_per_bin = hourly_gold['n'].to_numpy(dtype=float)

# Sum the game counts for each local hour
volume_by_local_hour = np.zeros(24)
for h, c in zip(local_hour, counts_per_bin):
    volume_by_local_hour[h] += c

plt.figure(figsize=(10, 5))
plt.bar(range(24), volume_by_local_hour, color='mediumorchid', alpha=0.8, edgecolor='black')
plt.title(f"Total Game Volume by Local Hour (UTC{offset_hours:+d}), PLATFORMID={platform}")
plt.xlabel("Local Hour of Day (0-23)")
plt.ylabel("Total Number of Games (Sum of 'n')")
plt.xticks(range(24))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

Also look at trends across the week

In [ ]:
%%time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Rebuild local-hour volume each run so partial reruns do not reuse a stale platform.
offset_hours = platform_utc_offset_hours(platform)
local_hour = np.floor((hourly_gold['hour_idx'] + offset_hours) % 24.0).astype(int)
counts_per_bin = hourly_gold['n'].to_numpy(dtype=float)
volume_by_local_hour = np.zeros(24)
for h, c in zip(local_hour, counts_per_bin):
    volume_by_local_hour[h] += c

# Convert hour_idx to datetime (hour_idx represents hours since Unix epoch)
hourly_gold['datetime_utc'] = pd.to_datetime(hourly_gold['hour_idx'] * 3600, unit='s')

# Extract day name and aggregate weekday volumes
hourly_gold['day_name'] = hourly_gold['datetime_utc'].dt.day_name()
day_volumes = hourly_gold.groupby('day_name')['n'].sum()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_volumes = day_volumes.reindex(day_order)

# Multi-panel engagement profile (hour-of-day + weekday)
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.55, 1.0]})

ax = axes[0]
hours = np.arange(24)
colors_hour = [COLORS["secondary"] if 7 <= h <= 23 else COLORS["accent"] for h in hours]
ax.bar(hours, volume_by_local_hour, color=colors_hour, alpha=0.9, edgecolor='white', linewidth=0.8)
ax.set_title(f"Game Volume by Local Hour (UTC{offset_hours:+d}), PLATFORMID={platform}")
ax.set_xlabel("Local Hour")
ax.set_ylabel("Total Games (sum of n)")
ax.set_xticks(np.arange(0, 24, 2))

peak_hour = int(np.argmax(volume_by_local_hour))
ax.axvline(peak_hour, color=COLORS["ink"], linestyle='--', linewidth=1.2, alpha=0.85)
ax.text(peak_hour + 0.2, ax.get_ylim()[1] * 0.94, f"Peak {peak_hour}:00", color=COLORS["ink"], fontsize=9)

ax = axes[1]
week_colors = [COLORS["primary"]] * 5 + [COLORS["highlight"], COLORS["accent"]]
ax.bar(day_volumes.index, day_volumes.values, color=week_colors, edgecolor='white', linewidth=0.9, alpha=0.9)
ax.set_title(f"Game Volume by Day of Week, PLATFORMID={platform}")
ax.set_xlabel("Day")
ax.set_ylabel("Total Games (sum of n)")
ax.tick_params(axis='x', rotation=30)

fig.suptitle(f"Engagement Rhythm Profile: {platform}", fontsize=13.5, fontweight='bold', color=COLORS["ink"])
plt.tight_layout()
plt.show()

Now: Can we identify players by their ID and follow them across the dataset? Find 10 players with a lot of games and log all the data for those games


In [ ]:
%%time
import pandas as pd
import numpy as np

conn = connect_db(db_path)

# Find top N players by game count on the current platform
top_players_query = """
SELECT ACCOUNTID, COUNT(*) as game_count
FROM riotData
WHERE PLATFORMID = ? AND ACCOUNTID IS NOT NULL
GROUP BY ACCOUNTID
ORDER BY game_count DESC
LIMIT ?;
"""
top_players = pd.read_sql_query(top_players_query, conn, params=[platform, int(TOP_N_PLAYERS)])
print(f"Top {len(top_players)} players on {platform} by game count:")
display(top_players)

# Extract all game rows for this cohort
if not top_players.empty:
    account_ids_selected = top_players['ACCOUNTID'].tolist()
    placeholders = ','.join(['?'] * len(account_ids_selected))

    player_data_query = f"""
    SELECT *
    FROM riotData
    WHERE PLATFORMID = ? AND ACCOUNTID IN ({placeholders})
    ORDER BY ACCOUNTID, TIMESTAMP;
    """

    params = [platform] + account_ids_selected
    top_players_data = pd.read_sql_query(player_data_query, conn, params=params)

    # delta_mmr is the next observed rating minus the current rating for the same player.
    # Positive values mean the player's rating increased after this observed game.
    top_players_data['TIMESTAMP'] = pd.to_numeric(top_players_data['TIMESTAMP'], errors='coerce')
    top_players_data['RATING'] = pd.to_numeric(top_players_data['RATING'], errors='coerce')
    top_players_data = top_players_data.sort_values(['ACCOUNTID', 'TIMESTAMP']).copy()
    top_players_data['next_rating'] = top_players_data.groupby('ACCOUNTID')['RATING'].shift(-1)
    top_players_data['delta_mmr'] = top_players_data['next_rating'] - top_players_data['RATING']

    # Backward-compatible aliases used in later cells
    account_ids_100 = account_ids_selected
    top_100_data = top_players_data
    account_ids_500 = account_ids_selected
    top_500_data = top_players_data
    top_100_players = top_players
    top_500_players = top_players

    print(f"\nExtracted {len(top_players_data)} total rows for these {len(account_ids_selected)} players.")
    print(f"Rows with usable delta_mmr: {top_players_data['delta_mmr'].notna().sum()}")
    print(f"Mean delta_mmr: {top_players_data['delta_mmr'].mean():.3f}")
    print("Mean delta_mmr by WINLOSE:")
    display(top_players_data.groupby('WINLOSE', dropna=False)['delta_mmr'].agg(['count', 'mean', 'median']).reset_index())
    display(top_players_data.head(15))
else:
    print("No players found for this platform.")

conn.close()


### Individual Player Periodograms
Calculating Lomb-Scargle periodograms for individual players to see if their performance cycles periodically (e.g., every 24 hours).

In [ ]:
%%time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle

# Convert columns to numeric before calculating to avoid string division errors
top_players_data['GOLD'] = pd.to_numeric(top_players_data['GOLD'], errors='coerce')
top_players_data['TIMEPLAYED'] = pd.to_numeric(top_players_data['TIMEPLAYED'], errors='coerce')
top_players_data['TIMESTAMP'] = pd.to_numeric(top_players_data['TIMESTAMP'], errors='coerce')

# Create a duration-normalized performance metric for individual games.
# Riot TIMEPLAYED is in seconds, so this is true gold per second.
top_players_data['gold_per_sec'] = top_players_data['GOLD'] / top_players_data['TIMEPLAYED'].where(top_players_data['TIMEPLAYED'] > 0)

unique_players = top_players_data['ACCOUNTID'].dropna().unique()
players_to_plot = unique_players[:10]
print(f"Plotting {len(players_to_plot)} representative players out of {len(unique_players)} in the selected cohort.")

# Prepare grid for plotting up to 10 players
fig, axes = plt.subplots(5, 2, figsize=(15, 18))
axes = axes.flatten()

# Periodogram setup
min_period_h = 6.0
max_period_h = 48.0
n_freq = 10000
min_freq = 1.0 / max_period_h
max_freq = 1.0 / min_period_h
frequency = np.linspace(min_freq, max_freq, n_freq)
period = 1.0 / frequency

for i, player_id in enumerate(players_to_plot):
    player_df = top_players_data[top_players_data['ACCOUNTID'] == player_id].copy()
    player_df = player_df.sort_values('TIMESTAMP')

    t_hours = player_df['TIMESTAMP'].to_numpy(dtype=float, copy=True) / 3600000.0
    t_hours = t_hours - t_hours.min()

    y = player_df['gold_per_sec'].to_numpy(dtype=float, copy=True)
    valid_mask = ~np.isnan(y)

    ax = axes[i]
    if np.sum(valid_mask) > 10:
        y_valid = y[valid_mask]
        t_hours_valid = t_hours[valid_mask]
        y_valid = y_valid - np.mean(y_valid)

        ls = LombScargle(t_hours_valid, y_valid)
        power = ls.power(frequency)

        best_idx = int(np.argmax(power))
        best_period = float(period[best_idx])

        ax.plot(period, power, color='steelblue')
        ax.axvline(24.0, color='red', linestyle='--', alpha=0.5, label='24h')
        ax.set_title(f'Player ID: {player_id}, PLATFORMID={platform} (n={len(y_valid)} games)\\nPeak Period: {best_period:.2f}h')
        ax.set_xlabel('Period (hours)')
        ax.set_ylabel('LS Power')
        ax.set_xlim(min_period_h, max_period_h)
        ax.grid(alpha=0.3)
        ax.legend()
    else:
        ax.set_title(f'Player ID: {player_id}, PLATFORMID={platform} (Not enough valid data)')
        ax.axis('off')

# Hide any unused axes if fewer than 10 players are available
for j in range(len(players_to_plot), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

### Incoherent Averaging of PC1 and PC2 Periodograms
By computing the Lomb-Scargle power for each player on the exact same frequency grid and then averaging those powers, we can perform an incoherent average. This now uses only the time-normalized PCA scores (`PC1` and `PC2`) rather than raw or single-variable performance metrics, so common periodicities are compared on the same PCA basis.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.timeseries import LombScargle
from joblib import Parallel, delayed

# PCA component scores use the hourly PCA basis from the earlier PCA cell.
pca_player_components = {
    'PC1': 'perf_factor_pc1',
    'PC2': 'perf_factor_pc2',
}
within_subject_metric_map = {
    'PC1': 'perf_factor_pc1',
    'PC2': 'perf_factor_pc2',
    'DeltaMMR': 'delta_mmr',
}
within_subject_metric_labels = {
    'PC1': 'PC1',
    'PC2': 'PC2',
    'DeltaMMR': 'Delta MMR',
}
PLAYER_PCA_BASIS_LABEL = 'hourly_time_normalized_pc1_pc2'

PLAYER_PCA_FEATURE_MAP = {
    'neutralcreep_per_min': 'NEUTRALCREEP',
    'enemycreep_per_min': 'ENEMYCREEP',
    'gold_per_min': 'GOLD',
    'damdealt_per_min': 'DAMDEALT',
    'kills_per_min': 'KILLS',
    'deaths_per_min': 'DEATHS',
    'assists_per_min': 'ASSISTS',
}


def ensure_player_pca_scores(player_df):
    """Project individual game rows onto the hourly PCA basis from Cell 21."""
    expected_scores = list(pca_player_components.values())
    has_current_scores = (
        all(col in player_df.columns for col in expected_scores)
        and player_df.attrs.get('perf_factor_pca_basis') == PLAYER_PCA_BASIS_LABEL
    )
    if has_current_scores:
        return player_df

    required_globals = ['numeric_cols', 'X_mean', 'X_std', 'Vt']
    missing_globals = [name for name in required_globals if name not in globals()]
    if missing_globals:
        raise RuntimeError(
            "Run the hourly PCA cell before player-level PCA projection. "
            f"Missing: {missing_globals}"
        )

    feature_names = list(numeric_cols)
    missing_feature_map = [
        col for col in feature_names
        if col != 'timedead_fraction' and col not in PLAYER_PCA_FEATURE_MAP
    ]
    if missing_feature_map:
        raise KeyError(f"No player-data mapping for PCA feature columns: {missing_feature_map}")

    source_cols = sorted(set(PLAYER_PCA_FEATURE_MAP.values()) | {'TIMEDEAD', 'TIMEPLAYED'})
    missing_source_cols = [col for col in source_cols if col not in player_df.columns]
    if missing_source_cols:
        raise KeyError(f"Missing PCA source columns in player data: {missing_source_cols}")

    work = player_df[source_cols].apply(pd.to_numeric, errors='coerce')
    timeplayed_minutes = (work['TIMEPLAYED'] / 60.0).where(work['TIMEPLAYED'] > 0)

    player_features = pd.DataFrame(index=work.index)
    for feature_col in feature_names:
        if feature_col == 'timedead_fraction':
            player_features[feature_col] = work['TIMEDEAD'] / work['TIMEPLAYED'].where(work['TIMEPLAYED'] > 0)
        else:
            source_col = PLAYER_PCA_FEATURE_MAP[feature_col]
            player_features[feature_col] = work[source_col] / timeplayed_minutes

    valid_mask = player_features.notna().all(axis=1)
    if valid_mask.sum() == 0:
        raise RuntimeError('No complete player rows available for PCA projection.')

    basis_mean = pd.Series(np.asarray(X_mean, dtype=float), index=feature_names)
    basis_std = pd.Series(np.asarray(X_std, dtype=float), index=feature_names).replace(0, 1.0)
    basis_vectors = np.asarray(Vt, dtype=float)

    if basis_vectors.shape[1] != len(feature_names):
        raise RuntimeError(
            "Hourly PCA basis does not match the current PCA feature columns. "
            "Rerun the hourly PCA cell."
        )

    X_player_z = (player_features.loc[valid_mask, feature_names] - basis_mean) / basis_std
    X_player_z = X_player_z.to_numpy(dtype=float, copy=True)

    result = player_df.copy()
    for component_index, (component_label, score_col) in enumerate(pca_player_components.items()):
        if component_index >= basis_vectors.shape[0]:
            raise RuntimeError(f"Hourly PCA basis does not contain {component_label}.")
        result[score_col] = np.nan
        result.loc[valid_mask, score_col] = X_player_z @ basis_vectors[component_index]

    result.attrs['perf_factor_pca_basis'] = PLAYER_PCA_BASIS_LABEL
    return result


def player_periodogram(player_df, score_col):
    player_df = player_df.dropna(subset=['TIMESTAMP', score_col]).sort_values('TIMESTAMP')
    if len(player_df) <= 10:
        return None

    t = player_df['TIMESTAMP'].to_numpy(dtype=float) / 3600000.0  # hours
    t = t - t.min()
    y = player_df[score_col].to_numpy(dtype=float)
    if np.nanstd(y) == 0:
        return None
    y = y - np.nanmean(y)

    try:
        return LombScargle(t, y).power(frequency)
    except Exception:
        return None


# Parameters
min_period = 6      # hours
max_period = 48     # hours
n_freq = 2000
frequency = np.linspace(1 / max_period, 1 / min_period, n_freq)
period = 1 / frequency

# Use the same top-player cohort, projected onto the hourly PCA basis, plus delta_mmr.
top_players_data = ensure_player_pca_scores(top_players_data)
missing_metric_cols = [col for col in within_subject_metric_map.values() if col not in top_players_data.columns]
if missing_metric_cols:
    raise KeyError(f"Missing within-subject metric columns: {missing_metric_cols}")

player_groups = [group for _, group in top_players_data.groupby('ACCOUNTID', sort=False)]

n_jobs = min(os.cpu_count() or 1, 8)
incoherent_results_by_component = {}

fig, axes = plt.subplots(
    1,
    len(within_subject_metric_map),
    figsize=(6.5 * len(within_subject_metric_map), 5),
    sharey=False,
)
axes = np.atleast_1d(axes)

for ax, (component_label, score_col) in zip(axes, within_subject_metric_map.items()):
    metric_label = within_subject_metric_labels.get(component_label, component_label)
    power_list = Parallel(n_jobs=n_jobs)(
        delayed(player_periodogram)(group, score_col)
        for group in player_groups
    )
    valid_powers = [power_array for power_array in power_list if power_array is not None]
    valid_players = len(valid_powers)

    if valid_players == 0:
        print(f'{metric_label}: no valid player periodograms.')
        ax.set_title(f'{metric_label}: no valid players, PLATFORMID={platform}')
        ax.set_xlabel('Period (hours)')
        continue

    mean_power_component = np.mean(valid_powers, axis=0)
    best_idx = np.argmax(mean_power_component)
    best_period_component = period[best_idx]

    for power_array in valid_powers:
        ax.plot(period, power_array, color=COLORS['muted'], alpha=0.08, linewidth=0.7)
    ax.plot(period, mean_power_component, color=COLORS['primary'], linewidth=2.5, label='Mean power')
    ax.axvline(best_period_component, color=COLORS['accent'], linestyle='--', linewidth=1.5, label=f'Peak = {best_period_component:.2f} h')
    ax.set_title(f'{metric_label} incoherent average ({valid_players:,} players), PLATFORMID={platform}')
    ax.set_xlabel('Period (hours)')
    ax.set_ylabel('Lomb-Scargle power')
    ax.legend()
    style_axes(ax)

    incoherent_results_by_component[component_label] = {
        'score_col': score_col,
        'mean_power': mean_power_component,
        'valid_players': valid_players,
        'best_period': best_period_component,
        'valid_powers': valid_powers,
    }

plt.suptitle(f'Per-player incoherent periodogram averages: PC1, PC2, and DeltaMMR, PLATFORMID={platform}', y=1.03)
plt.tight_layout()
plt.show()

# Backward-compatible aliases point to PC1 for older downstream cells.
mean_power = incoherent_results_by_component['PC1']['mean_power']
valid_players = incoherent_results_by_component['PC1']['valid_players']
best_period = incoherent_results_by_component['PC1']['best_period']

for component_label, result in incoherent_results_by_component.items():
    metric_label = within_subject_metric_labels.get(component_label, component_label)
    print(f"{metric_label}: peak average period = {result['best_period']:.2f} hours using {result['valid_players']} players")


### Incoherent Averaging for Selected Cohort
Reuse the same player cohort selected by TOP_N_PLAYERS and average per-player periodograms for `PC1` and `PC2` only, keeping the downstream analyses consistent with the PCA performance signals.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from astropy.timeseries import LombScargle

if 'ensure_player_pca_scores' not in globals():
    raise RuntimeError("Run the previous player PCA helper cell before this selected-cohort analysis.")
if 'within_subject_metric_map' not in globals():
    within_subject_metric_map = {'PC1': 'perf_factor_pc1', 'PC2': 'perf_factor_pc2'}
    within_subject_metric_labels = {'PC1': 'PC1', 'PC2': 'PC2'}


def compute_player_periodogram(group, score_col):
    group = group.dropna(subset=['TIMESTAMP', score_col]).sort_values('TIMESTAMP')
    if len(group) < 10:
        return None

    time_hours = group['TIMESTAMP'].to_numpy(dtype=float) / 3600000.0
    time_hours = time_hours - time_hours.min()
    values = group[score_col].to_numpy(dtype=float)
    if np.nanstd(values) == 0:
        return None
    values = values - values.mean()

    try:
        return LombScargle(time_hours, values).power(frequency)
    except Exception:
        return None


# Periodogram frequency grid
min_period = 6
max_period = 48
n_freq = 2000
frequency = np.linspace(1 / max_period, 1 / min_period, n_freq)
period = 1 / frequency

cohort_df = ensure_player_pca_scores(top_players_data.copy())
selected_players = cohort_df['ACCOUNTID'].dropna().unique()
print(f"Computing within-subject periodograms for {len(selected_players):,} selected players...")

player_groups = [
    group
    for _, group in cohort_df.groupby('ACCOUNTID', sort=False)
]

n_jobs = min(os.cpu_count() or 1, 8)
incoherent_results_selected_by_component = {}

fig, axes = plt.subplots(
    1,
    len(within_subject_metric_map),
    figsize=(6.5 * len(within_subject_metric_map), 5),
    sharey=False,
)
axes = np.atleast_1d(axes)

for ax, (component_label, score_col) in zip(axes, within_subject_metric_map.items()):
    metric_label = within_subject_metric_labels.get(component_label, component_label)
    results = Parallel(n_jobs=n_jobs)(
        delayed(compute_player_periodogram)(group, score_col)
        for group in player_groups
    )
    valid_powers_component = [power_array for power_array in results if power_array is not None]
    valid_players_component = len(valid_powers_component)

    if valid_players_component == 0:
        print(f'{metric_label}: no valid periodograms computed.')
        ax.set_title(f'{metric_label}: no valid players, PLATFORMID={platform}')
        ax.set_xlabel('Period (hours)')
        continue

    mean_power_component = np.mean(valid_powers_component, axis=0)
    best_idx = int(np.argmax(mean_power_component))
    best_period_component = period[best_idx]
    best_power_component = mean_power_component[best_idx]

    ax.plot(period, mean_power_component, color=COLORS['primary'], linewidth=2.5, label='Mean power')
    ax.axvline(best_period_component, color=COLORS['accent'], linestyle='--', linewidth=1.5, label=f'Peak = {best_period_component:.2f} h')
    ax.set_title(f'{metric_label}: {valid_players_component:,} players, PLATFORMID={platform}')
    ax.set_xlabel('Period (hours)')
    ax.set_ylabel('Mean Lomb-Scargle power')
    ax.legend()
    style_axes(ax)

    incoherent_results_selected_by_component[component_label] = {
        'score_col': score_col,
        'valid_powers': valid_powers_component,
        'valid_players': valid_players_component,
        'mean_power': mean_power_component,
        'best_period': best_period_component,
        'best_power': best_power_component,
    }

plt.suptitle(f'Incoherent averaged periodograms for selected players: PC1, PC2, and DeltaMMR, PLATFORMID={platform}', y=1.03)
plt.tight_layout()
plt.show()

for component_label, result in incoherent_results_selected_by_component.items():
    metric_label = within_subject_metric_labels.get(component_label, component_label)
    print(f"{metric_label}: valid players = {result['valid_players']:,}")
    print(f"{metric_label}: strongest period = {result['best_period']:.2f} hours")
    print(f"{metric_label}: peak mean power = {result['best_power']:.4f}\n")

# Backward-compatible aliases point to PC1 for older downstream cells.
pc1_selected = incoherent_results_selected_by_component['PC1']
valid_powers = pc1_selected['valid_powers']
valid_players_selected = pc1_selected['valid_players']
mean_power_selected = pc1_selected['mean_power']
best_period_selected = pc1_selected['best_period']
best_power_selected = pc1_selected['best_power']

# Additional aliases for older notebook text that still refers to the top-100 periodogram.
mean_power_100 = mean_power_selected
valid_players_100 = valid_players_selected
best_period_100 = best_period_selected
best_power_100 = best_power_selected


### Phase Extraction for 24-Hour Cycles
Using our OLS sinusoid fit to extract the exact hour of the day (phase) when each player's performance metric peaks.

In [ ]:
%%time
import os
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if 'top_players_data' not in globals() or top_players_data is None or len(top_players_data) == 0:
    raise RuntimeError("Expected top_players_data from the cohort extraction cell.")
if 'ensure_player_pca_scores' not in globals():
    raise RuntimeError("Run the player PCA helper cell before phase extraction.")
if 'within_subject_metric_map' not in globals():
    within_subject_metric_map = {'PC1': 'perf_factor_pc1', 'PC2': 'perf_factor_pc2'}
    within_subject_metric_labels = {'PC1': 'PC1', 'PC2': 'PC2'}

offset_hours_local = int(platform_utc_offset_hours(platform))

# Ensure PC1 and PC2 are projections onto the same hourly PCA basis used earlier.
top_players_data = ensure_player_pca_scores(top_players_data)


def benjamini_hochberg_mask(p_values, alpha=0.05):
    """Return a boolean mask for Benjamini-Hochberg FDR significance."""
    p_values = np.asarray(p_values, dtype=float)
    keep = np.zeros(len(p_values), dtype=bool)
    finite_mask = np.isfinite(p_values)
    finite_p = p_values[finite_mask]

    if len(finite_p) == 0:
        return keep

    order = np.argsort(finite_p)
    sorted_p = finite_p[order]
    ranks = np.arange(1, len(sorted_p) + 1)
    passed = sorted_p <= alpha * ranks / len(sorted_p)

    if not np.any(passed):
        return keep

    threshold = sorted_p[np.max(np.where(passed))]
    keep[finite_mask] = finite_p <= threshold
    return keep


def player_phase(player_df, score_col):
    player_df = player_df.dropna(subset=['TIMESTAMP', score_col]).sort_values('TIMESTAMP')
    if len(player_df) <= 10:
        return None

    t_utc_hours = player_df['TIMESTAMP'].to_numpy(dtype=float, copy=True) / 3600000.0
    y = player_df[score_col].to_numpy(dtype=float, copy=True)
    if np.nanstd(y) == 0:
        return None

    res = fit_sinusoid_ols(
        t_hours=t_utc_hours,
        y=y,
        period_h=24.0,
        fit_intercept=True,
        robust=False,
    )
    peak_utc = res['lag_h_mod_period']
    peak_local = (peak_utc + offset_hours_local) % 24.0
    return (peak_local, res['amp'], res['p_joint'])

n_jobs = max(1, (os.cpu_count() or 1) - 1)
print(f"Computing per-player phases in parallel with n_jobs={n_jobs}...")

phase_alpha = 0.05
phase_results_by_component = {}
fig = plt.figure(figsize=(14.5, 5.2 * len(within_subject_metric_map)))

for row_idx, (component_label, score_col) in enumerate(within_subject_metric_map.items()):
    metric_label = within_subject_metric_labels.get(component_label, component_label)
    cohort_phase_df = top_players_data.copy()
    cohort_phase_df['TIMESTAMP'] = pd.to_numeric(cohort_phase_df['TIMESTAMP'], errors='coerce')
    cohort_phase_df[score_col] = pd.to_numeric(cohort_phase_df[score_col], errors='coerce')
    cohort_phase_df = cohort_phase_df.dropna(subset=['ACCOUNTID', 'TIMESTAMP', score_col]).copy()

    player_groups = [g for _, g in cohort_phase_df.groupby('ACCOUNTID', sort=False)]
    phase_results = Parallel(n_jobs=n_jobs, prefer='threads')(
        delayed(player_phase)(g, score_col) for g in player_groups
    )

    phases_local_component = []
    amplitudes_component = []
    p_values_component = []
    for item in phase_results:
        if item is None:
            continue
        peak_local, amp, p_joint = item
        phases_local_component.append(peak_local)
        amplitudes_component.append(amp)
        p_values_component.append(p_joint)

    phases_local_component = np.array(phases_local_component)
    amplitudes_component = np.array(amplitudes_component)
    p_values_component = np.array(p_values_component)
    nominal_sig_mask = p_values_component < phase_alpha
    fdr_sig_mask = benjamini_hochberg_mask(p_values_component, alpha=phase_alpha)
    phases_sig_component = phases_local_component[fdr_sig_mask]

    phase_results_by_component[component_label] = {
        'score_col': score_col,
        'metric_label': metric_label,
        'phases_local': phases_local_component,
        'amplitudes': amplitudes_component,
        'p_values': p_values_component,
        'nominal_sig_mask': nominal_sig_mask,
        'fdr_sig_mask': fdr_sig_mask,
        'phases_sig': phases_sig_component,
        'alpha': phase_alpha,
        'correction': 'Benjamini-Hochberg FDR',
    }

    nominal_count = int(np.sum(nominal_sig_mask))
    fdr_count = int(np.sum(fdr_sig_mask))
    print(f"\n{metric_label}:")
    print(f"  Total players analyzed: {len(phases_local_component)}")
    print(f"  Nominal p < {phase_alpha}: {nominal_count}")
    print(f"  FDR-significant 24h cycles ({phase_results_by_component[component_label]['correction']}, q < {phase_alpha}): {fdr_count}")
    print(f"  Using timezone offset: UTC{offset_hours_local:+d}")

    ax1 = fig.add_subplot(len(within_subject_metric_map), 2, 2 * row_idx + 1)
    ax2 = fig.add_subplot(len(within_subject_metric_map), 2, 2 * row_idx + 2, projection='polar')

    bins = np.arange(0, 25, 1)
    ax1.hist(phases_local_component, bins=bins, color=COLORS["secondary"], edgecolor='white', alpha=0.55, label='All analyzed')
    ax1.hist(phases_sig_component, bins=bins, color=COLORS["accent"], edgecolor='white', alpha=0.88, label='FDR-significant')
    ax1.set_title(f"{metric_label} Peak Hour Distribution ({platform})")
    ax1.set_xlabel("Local Hour of Day")
    ax1.set_ylabel("Players")
    ax1.set_xticks(np.arange(0, 25, 2))
    style_axes(ax1, grid_axis='y')
    ax1.legend(frameon=False)

    if len(phases_sig_component) > 0:
        theta_sig = (phases_sig_component / 24.0) * 2.0 * np.pi
        polar_bins = np.linspace(0, 2.0 * np.pi, 25)
        counts, edges = np.histogram(theta_sig, bins=polar_bins)
        widths = np.diff(edges)
        ax2.bar(edges[:-1], counts, width=widths, align='edge', color=COLORS["primary"], alpha=0.85, edgecolor='white')

    ax2.set_theta_zero_location('N')
    ax2.set_theta_direction(-1)
    ax2.set_xticks(np.linspace(0, 2 * np.pi, 8, endpoint=False))
    ax2.set_xticklabels(['00', '03', '06', '09', '12', '15', '18', '21'])
    ax2.set_title(f"{metric_label} FDR-Significant Peaks on 24 h Clock ({platform})", va='bottom')
    ax2.grid(alpha=0.25)

fig.suptitle(
    f"Circadian Phase Landscape (Selected Cohort)\nPLATFORMID={platform}, Metrics: PC1, PC2, and DeltaMMR",
    fontsize=13.5,
    fontweight='bold',
    color=COLORS["ink"],
)
plt.tight_layout()
plt.show()

# Backward-compatible aliases for PC1-only downstream cells.
pc1_phase_results = phase_results_by_component['PC1']
phases_local = pc1_phase_results['phases_local']
amplitudes = pc1_phase_results['amplitudes']
p_values = pc1_phase_results['p_values']
phases_sig = pc1_phase_results['phases_sig']
phases_local_500 = phases_local
p_values_500 = p_values

# Explicit PC2 and DeltaMMR aliases for quick notebook inspection.
pc2_phase_results = phase_results_by_component.get('PC2')
if pc2_phase_results is not None:
    phases_local_pc2 = pc2_phase_results['phases_local']
    amplitudes_pc2 = pc2_phase_results['amplitudes']
    p_values_pc2 = pc2_phase_results['p_values']
    phases_sig_pc2 = pc2_phase_results['phases_sig']

delta_mmr_phase_results = phase_results_by_component.get('DeltaMMR')
if delta_mmr_phase_results is not None:
    phases_local_delta_mmr = delta_mmr_phase_results['phases_local']
    amplitudes_delta_mmr = delta_mmr_phase_results['amplitudes']
    p_values_delta_mmr = delta_mmr_phase_results['p_values']
    phases_sig_delta_mmr = delta_mmr_phase_results['phases_sig']


### Cohort Note
The selected-cohort phase extraction above already scales to TOP_N_PLAYERS, so no separate top-500 query step is needed.

In [ ]:
%%time
# No-op by design: player extraction is centralized in the earlier cohort cell.
print(
    f"Skipping duplicate extraction cell. Using selected cohort from earlier step "
    f"(TOP_N_PLAYERS={TOP_N_PLAYERS}, rows={len(top_players_data) if 'top_players_data' in globals() else 0})."
)

In [ ]:
%%time
# No-op by design: phase extraction is centralized in the selected-cohort phase cell.
if 'phase_results_by_component' in globals():
    counts_text = ", ".join(
        f"{label}: n={len(result['phases_local'])}, fdr_significant={len(result['phases_sig'])}"
        for label, result in phase_results_by_component.items()
    )
    print(f"Skipping duplicate phase extraction cell. Reusing selected-cohort phase arrays ({counts_text}).")
else:
    print(
        f"Skipping duplicate phase extraction cell. Reusing selected-cohort phase arrays "
        f"(n={len(phases_local) if 'phases_local' in globals() else 0}, "
        f"fdr_significant={len(phases_sig) if 'phases_sig' in globals() else 0})."
    )

In [ ]:
%%time
# Dependency check for circular modality analysis
try:
    from scipy.stats import vonmises  # noqa: F401
    print("scipy.stats.vonmises is available in the active environment.")
except ImportError:
    raise ImportError(
        "SciPy is required for circular von Mises modeling. "
        "For this project use uv: uv pip install --python .venv/bin/python scipy"
    )

### Owls vs Larks: Testing for Circular Multimodality
Using one- and two-component circular von Mises models to test whether FDR-significant player peak times separate into distinct circadian preference groups.


In [ ]:
%%time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import vonmises

if 'phase_results_by_component' in globals():
    phase_sources = {
        component_label: result['phases_sig']
        for component_label, result in phase_results_by_component.items()
    }
elif 'phases_sig' in globals():
    phase_sources = {'PC1': phases_sig}
else:
    raise RuntimeError("Expected phase results from the selected-cohort phase cell.")


def _wrap_mu(mu):
    return np.mod(mu, 2.0 * np.pi)


def _kappa_from_rbar(rbar):
    # Standard approximation for von Mises concentration from mean resultant length.
    rbar = float(np.clip(rbar, 1e-8, 0.999999))
    if rbar < 0.53:
        return 2 * rbar + rbar**3 + 5 * (rbar**5) / 6
    if rbar < 0.85:
        return -0.4 + 1.39 * rbar + 0.43 / (1 - rbar)
    return 1 / (rbar**3 - 4 * rbar**2 + 3 * rbar)


def fit_vonmises_1comp(theta_vals):
    z = np.exp(1j * theta_vals)
    z_mean = np.mean(z)
    mu = _wrap_mu(np.angle(z_mean))
    rbar = np.abs(z_mean)
    kappa = max(1e-4, _kappa_from_rbar(rbar))
    ll = np.sum(vonmises.logpdf(theta_vals, kappa, loc=mu))
    # Params: mu, kappa
    bic = 2 * np.log(len(theta_vals)) - 2 * ll
    return {"mu": mu, "kappa": kappa, "loglik": ll, "bic": bic}


def fit_vonmises_2comp(theta_vals, max_iter=200, tol=1e-6):
    n = len(theta_vals)

    # Stable init: opposite means on circle, shared concentration from global Rbar.
    z = np.exp(1j * theta_vals)
    mu_global = _wrap_mu(np.angle(np.mean(z)))
    rbar_global = np.abs(np.mean(z))
    kappa_global = max(1e-4, _kappa_from_rbar(rbar_global))

    pi1 = 0.5
    mu1 = _wrap_mu(mu_global)
    mu2 = _wrap_mu(mu_global + np.pi)
    kappa1 = kappa_global
    kappa2 = kappa_global

    prev_ll = -np.inf

    for _ in range(max_iter):
        f1 = vonmises.pdf(theta_vals, kappa1, loc=mu1)
        f2 = vonmises.pdf(theta_vals, kappa2, loc=mu2)

        mix = pi1 * f1 + (1 - pi1) * f2
        mix = np.clip(mix, 1e-300, None)
        ll = np.sum(np.log(mix))

        # E-step
        gamma1 = (pi1 * f1) / mix
        gamma2 = 1.0 - gamma1

        # M-step
        pi1 = float(np.clip(np.mean(gamma1), 1e-4, 1 - 1e-4))

        z1 = np.sum(gamma1 * np.exp(1j * theta_vals))
        z2 = np.sum(gamma2 * np.exp(1j * theta_vals))

        mu1 = _wrap_mu(np.angle(z1))
        mu2 = _wrap_mu(np.angle(z2))

        rbar1 = np.abs(z1) / np.sum(gamma1)
        rbar2 = np.abs(z2) / np.sum(gamma2)

        kappa1 = max(1e-4, _kappa_from_rbar(rbar1))
        kappa2 = max(1e-4, _kappa_from_rbar(rbar2))

        if np.abs(ll - prev_ll) < tol:
            break
        prev_ll = ll

    # Params: pi1, mu1, kappa1, mu2, kappa2
    bic = 5 * np.log(n) - 2 * ll
    return {
        "pi1": pi1,
        "pi2": 1 - pi1,
        "mu1": mu1,
        "mu2": mu2,
        "kappa1": kappa1,
        "kappa2": kappa2,
        "loglik": ll,
        "bic": bic,
    }


circular_results_by_component = {}
fig, axes = plt.subplots(len(phase_sources), 1, figsize=(10, 4.8 * len(phase_sources)))
axes = np.atleast_1d(axes)

for ax, (component_label, component_hours) in zip(axes, phase_sources.items()):
    metric_label = phase_results_by_component.get(component_label, {}).get('metric_label', component_label) if 'phase_results_by_component' in globals() else component_label
    data_hours_component = np.asarray(component_hours, dtype=float).copy()
    theta = (data_hours_component / 24.0) * 2.0 * np.pi

    print(f"--- Circular Modality Test on {metric_label}: {len(data_hours_component)} FDR-Significant Players ---\n")

    if len(theta) < 20:
        print(f"Skipping {metric_label}: need at least 20 FDR-significant players for stable circular mixture fitting.")
        ax.text(0.5, 0.5, f"{metric_label}: not enough FDR-significant players", ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        continue

    vm1_component = fit_vonmises_1comp(theta)
    vm2_component = fit_vonmises_2comp(theta)

    circular_results_by_component[component_label] = {
        'data_hours': data_hours_component,
        'theta': theta,
        'vm1': vm1_component,
        'vm2': vm2_component,
        'metric_label': metric_label,
    }

    print("Circular von Mises Model Comparison:")
    print(f"  1 Component BIC: {vm1_component['bic']:.2f}")
    print(f"  2 Components BIC: {vm2_component['bic']:.2f}")
    if vm2_component['bic'] < vm1_component['bic']:
        print("  Result: 2-component circular model preferred (is consistent with an Owls/Larks grouping).")
    else:
        print("  Result: 1-component circular model preferred.")

    mu1_h = (vm2_component['mu1'] / (2 * np.pi)) * 24.0
    mu2_h = (vm2_component['mu2'] / (2 * np.pi)) * 24.0
    print(f"  Component peak hours (local): ~{mu1_h:.2f}h and ~{mu2_h:.2f}h")

    styled_hist(
        ax,
        data_hours_component,
        bins=np.arange(0, 25, 1),
        density=True,
        variant="accent",
        alpha=0.55,
        edgecolor='white',
        label='Empirical Data (FDR-significant)',
    )

    x_hours = np.linspace(0, 24, 1000)
    x_theta = (x_hours / 24.0) * 2.0 * np.pi

    # Convert angular pdf to hour-domain density via dtheta/dh = 2pi/24.
    scale = (2.0 * np.pi / 24.0)
    y_vm1 = vonmises.pdf(x_theta, vm1_component['kappa'], loc=vm1_component['mu']) * scale
    y_vm2 = (
        vm2_component['pi1'] * vonmises.pdf(x_theta, vm2_component['kappa1'], loc=vm2_component['mu1'])
        + vm2_component['pi2'] * vonmises.pdf(x_theta, vm2_component['kappa2'], loc=vm2_component['mu2'])
    ) * scale

    ax.plot(x_hours, y_vm1, color=COLORS["ink"], linestyle='--', linewidth=2, label='1-Component von Mises')
    ax.plot(x_hours, y_vm2, color=COLORS["primary"], linewidth=2, label='2-Component von Mises')
    ax.set_title(f"Circular Modality Test ('Owls vs Larks') - PLATFORMID={platform}\nMetric: {metric_label} Peak Local Hour")
    ax.set_xlabel("Local Peak Performance Hour (0-23)")
    ax.set_ylabel("Density")
    ax.legend(loc='upper right', frameon=False)
    ax.set_xlim(0, 24)
    style_axes(ax, grid_axis='y')

if not circular_results_by_component:
    raise RuntimeError("No metric had enough significant players for circular mixture fitting.")

# Backward-compatible aliases for PC1-only downstream cells.
preferred_component = 'PC1' if 'PC1' in circular_results_by_component else next(iter(circular_results_by_component))
preferred_result = circular_results_by_component[preferred_component]
data_hours = preferred_result['data_hours']
vm1 = preferred_result['vm1']
vm2 = preferred_result['vm2']

plt.tight_layout()
plt.show()

### Export Results
Save critical tables and figures into a server-specific folder on Google Drive.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Define output directory based on environment and platform/server
if os.path.exists('/content/drive/Shareddrives/MSc_2026_Riot/results'):
    base_dir = '/content/drive/Shareddrives/MSc_2026_Riot/results'
else:
    # Local fallback for teaching/demo runs
    base_dir = os.environ.get('RIOT_RESULTS_DIR', './results')

output_dir = f"{base_dir}/{platform}"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory ready at: {output_dir}")

# 1. Save Critical Tables
# Hourly Overview
if 'hourly_overview' in globals():
    hourly_overview.to_csv(f"{output_dir}/hourly_overview_{platform}.csv", index=False)

# Top players list from selected cohort
if 'top_players' in globals():
    top_players.to_csv(f"{output_dir}/top_players_{platform}.csv", index=False)

# FDR-significant Player Phases (for Owls vs Larks)
if 'phase_results_by_component' in globals():
    for component_label, result in phase_results_by_component.items():
        phase_table = pd.DataFrame({'phase_local_peak': result['phases_sig']})
        phase_table.insert(0, 'component', component_label)
        phase_table['metric_label'] = result.get('metric_label', component_label)
        phase_table['correction'] = result.get('correction', 'Benjamini-Hochberg FDR')
        phase_table['alpha'] = result.get('alpha', 0.05)
        phase_table.to_csv(f"{output_dir}/significant_phases_{component_label.lower()}_{platform}.csv", index=False)

    # Preserve the original PC1 filename for older notebooks/scripts.
    if 'PC1' in phase_results_by_component:
        pc1_result = phase_results_by_component['PC1']
        pc1_phase_table = pd.DataFrame({'phase_local_peak': pc1_result['phases_sig']})
        pc1_phase_table['correction'] = pc1_result.get('correction', 'Benjamini-Hochberg FDR')
        pc1_phase_table['alpha'] = pc1_result.get('alpha', 0.05)
        pc1_phase_table.to_csv(
            f"{output_dir}/significant_phases_{platform}.csv",
            index=False,
        )
elif 'phases_sig' in globals():
    pd.DataFrame({'phase_local_peak': phases_sig}).to_csv(f"{output_dir}/significant_phases_{platform}.csv", index=False)

print("Critical tables saved as CSVs.")

In [ ]:
# 2. Re-plot and save the final circular mixture figures
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import vonmises

if 'circular_results_by_component' in globals() and circular_results_by_component:
    os.makedirs(output_dir, exist_ok=True)

    for component_label, result in circular_results_by_component.items():
        data_hours_component = result['data_hours']
        metric_label = result.get('metric_label', component_label)
        vm1_component = result['vm1']
        vm2_component = result['vm2']

        x_hours = np.linspace(0, 24, 1000)
        x_theta = (x_hours / 24.0) * 2.0 * np.pi
        scale = (2.0 * np.pi / 24.0)

        y_vm1 = vonmises.pdf(x_theta, vm1_component['kappa'], loc=vm1_component['mu']) * scale
        y_vm2 = (
            vm2_component['pi1'] * vonmises.pdf(x_theta, vm2_component['kappa1'], loc=vm2_component['mu1'])
            + vm2_component['pi2'] * vonmises.pdf(x_theta, vm2_component['kappa2'], loc=vm2_component['mu2'])
        ) * scale

        fig = plt.figure(figsize=(14.2, 5.6))
        ax1 = fig.add_subplot(1, 2, 1)
        ax2 = fig.add_subplot(1, 2, 2, projection='polar')

        ax1.hist(
            data_hours_component,
            bins=np.arange(0, 25, 1),
            density=True,
            color=COLORS["secondary"],
            edgecolor='white',
            alpha=0.5,
            label='Empirical (FDR-significant phases)',
        )
        ax1.plot(x_hours, y_vm1, linestyle='--', linewidth=2.0, color=COLORS["ink"], label='1-component von Mises')
        ax1.plot(x_hours, y_vm2, linewidth=2.4, color=COLORS["accent"], label='2-component von Mises')
        ax1.set_title(f"{metric_label} Hour-Domain Density Fit ({platform})")
        ax1.set_xlabel("Local Peak Performance Hour")
        ax1.set_ylabel("Density")
        ax1.set_xlim(0, 24)
        style_axes(ax1, grid_axis='y')
        ax1.legend(frameon=False, loc='upper right')

        theta_data = (data_hours_component / 24.0) * 2.0 * np.pi
        polar_bins = np.linspace(0, 2.0 * np.pi, 25)
        polar_counts, polar_edges = np.histogram(theta_data, bins=polar_bins, density=True)
        ax2.bar(
            polar_edges[:-1],
            polar_counts,
            width=np.diff(polar_edges),
            align='edge',
            color=COLORS["primary"],
            alpha=0.75,
            edgecolor='white',
            linewidth=0.8,
        )

        # Overlay two-component fit in polar coordinates.
        ax2.plot(x_theta, y_vm2, color=COLORS["accent"], linewidth=2.2)
        ax2.plot(x_theta, y_vm1, color=COLORS["ink"], linewidth=1.8, linestyle='--')
        ax2.set_theta_zero_location('N')
        ax2.set_theta_direction(-1)
        ax2.set_xticks(np.linspace(0, 2 * np.pi, 8, endpoint=False))
        ax2.set_xticklabels(['00', '03', '06', '09', '12', '15', '18', '21'])
        ax2.set_title(f"{metric_label} Circular Density View ({platform})", va='bottom')
        ax2.grid(alpha=0.22)

        fig.suptitle(
            f"Owls vs Larks Circular Modality: {platform}\nMetric: {metric_label} Peak Local Hour",
            fontsize=13.5,
            fontweight='bold',
            color=COLORS["ink"],
        )

        plt.tight_layout()
        fig_path = f"{output_dir}/owls_vs_larks_circular_vm_{component_label.lower()}_{platform}.png"
        plt.savefig(fig_path, dpi=320, bbox_inches='tight')
        print(f"Figure successfully saved to {fig_path}")

        # Preserve the original PC1 filename for older notebooks/scripts.
        if component_label == 'PC1':
            legacy_fig_path = f"{output_dir}/owls_vs_larks_circular_vm_{platform}.png"
            plt.savefig(legacy_fig_path, dpi=320, bbox_inches='tight')
            print(f"Legacy PC1 figure also saved to {legacy_fig_path}")

        plt.show()
elif all(v in globals() for v in ['data_hours', 'vm1', 'vm2']):
    print("Only legacy PC1 circular mixture variables were found; rerun the circular modality cell to save all metric figures.")
else:
    print("No circular mixture results available to save.")